In [72]:
# ============================================================
# R2-01 — CLEAN RUNTIME CHECK
# ============================================================

import os
import sys
import platform
from pathlib import Path

print("=" * 80)
print("R2 — STRICT FEASIBILITY / VALIDITY AUDIT")
print("STEP R2-01 — CLEAN RUNTIME CHECK")
print("=" * 80)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

print("\n/content contents:")
for p in sorted(Path("/content").iterdir()):
    print(" ", p.name)

print("\n✓ Fresh R2 runtime ready")

R2 — STRICT FEASIBILITY / VALIDITY AUDIT
STEP R2-01 — CLEAN RUNTIME CHECK
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content

/content contents:
  .config
  XAgentic_Final_Paper_Evidence_2026-10-06 (1).zip
  XAgentic_R1_Phrase_Generalization_FINAL_2026-10-07.zip
  revision_experiments
  sample_data
  xagentic_r1_work

✓ Fresh R2 runtime ready


In [73]:
# ============================================================
# R2-02 — CREATE ISOLATED R2 WORKSPACE
# ============================================================

from pathlib import Path

ROOT = Path("/content/xagentic_r2_work")

DIRS = {
    "input": ROOT / "00_INPUT_FROZEN",
    "source": ROOT / "01_SOURCE",
    "audit": ROOT / "02_R2_AUDIT",
    "results_raw": ROOT / "03_RESULTS_RAW",
    "results_agg": ROOT / "04_RESULTS_AGGREGATE",
    "evidence": ROOT / "05_FINAL_EVIDENCE",
    "docs": ROOT / "06_DOCUMENTATION",
    "archive": ROOT / "07_ARCHIVE",
}

# Safety: R2 must not overwrite an earlier R2 workspace.
if ROOT.exists():
    existing = list(ROOT.rglob("*"))
    if existing:
        raise RuntimeError(
            f"STOP: Existing non-empty R2 workspace found: {ROOT}\n"
            "Do not overwrite it."
        )

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("R2-02 — ISOLATED WORKSPACE CREATED")
print("=" * 80)

for name, path in DIRS.items():
    print(f"{name:15s}: {path}")

print("\nIMPORTANT:")
print("Existing R1/E1-E5 material remains untouched.")
print("All new R2 outputs will be written only under:")
print(ROOT)

print("\n✓ R2 isolated workspace ready")

R2-02 — ISOLATED WORKSPACE CREATED
input          : /content/xagentic_r2_work/00_INPUT_FROZEN
source         : /content/xagentic_r2_work/01_SOURCE
audit          : /content/xagentic_r2_work/02_R2_AUDIT
results_raw    : /content/xagentic_r2_work/03_RESULTS_RAW
results_agg    : /content/xagentic_r2_work/04_RESULTS_AGGREGATE
evidence       : /content/xagentic_r2_work/05_FINAL_EVIDENCE
docs           : /content/xagentic_r2_work/06_DOCUMENTATION
archive        : /content/xagentic_r2_work/07_ARCHIVE

IMPORTANT:
Existing R1/E1-E5 material remains untouched.
All new R2 outputs will be written only under:
/content/xagentic_r2_work

✓ R2 isolated workspace ready


In [74]:
# ============================================================
# R2-03 — FINGERPRINT SOURCE EVIDENCE PACKAGES
# ============================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

ROOT = Path("/content/xagentic_r2_work")
INPUT = ROOT / "00_INPUT_FROZEN"

R1_ZIP = Path(
    "/content/XAgentic_R1_Phrase_Generalization_FINAL_2026-10-07.zip"
)

BASE_ZIP = Path(
    "/content/XAgentic_Final_Paper_Evidence_2026-10-06 (1).zip"
)

print("=" * 100)
print("R2-03 — SOURCE PACKAGE FINGERPRINT")
print("=" * 100)

assert R1_ZIP.exists(), f"STOP: Missing R1 package: {R1_ZIP}"
assert BASE_ZIP.exists(), f"STOP: Missing base evidence package: {BASE_ZIP}"

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

packages = {
    "R1_FINAL": R1_ZIP,
    "BASE_E1_E5": BASE_ZIP,
}

manifest = {}

for label, path in packages.items():
    digest = sha256_file(path)

    manifest[label] = {
        "filename": path.name,
        "original_path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
    }

    print(f"\n[{label}]")
    print("File   :", path.name)
    print("Size   :", f"{path.stat().st_size:,}", "bytes")
    print("SHA256 :", digest)

manifest["created_utc"] = datetime.now(timezone.utc).isoformat()
manifest["purpose"] = "R2 strict feasibility / validity audit provenance"

manifest_path = INPUT / "R2_INPUT_PACKAGE_MANIFEST.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nManifest:")
print(manifest_path)

print("\n✓ Both source packages fingerprinted")
print("✓ No source package modified")

R2-03 — SOURCE PACKAGE FINGERPRINT

[R1_FINAL]
File   : XAgentic_R1_Phrase_Generalization_FINAL_2026-10-07.zip
Size   : 5,403,252 bytes
SHA256 : bdac27c42a7514700dc878c372c7a2555dd6a1bfcaca9326b1df9900a2e371a4

[BASE_E1_E5]
File   : XAgentic_Final_Paper_Evidence_2026-10-06 (1).zip
Size   : 384,861 bytes
SHA256 : 04a9c98f6cfdf89132fe70f26b7aec5e011805255cd4d202726674d09b8c601a

Manifest:
/content/xagentic_r2_work/00_INPUT_FROZEN/R2_INPUT_PACKAGE_MANIFEST.json

✓ Both source packages fingerprinted
✓ No source package modified


In [75]:
# ============================================================
# R2-04 — COPY SOURCE PACKAGES INTO R2 FROZEN INPUT
# ============================================================

import shutil

print("=" * 100)
print("R2-04 — FREEZE SOURCE PACKAGES")
print("=" * 100)

frozen_packages = {}

for label, source in packages.items():

    destination = INPUT / source.name

    shutil.copy2(source, destination)

    src_hash = sha256_file(source)
    dst_hash = sha256_file(destination)

    print(f"\n[{label}]")
    print("Source :", source)
    print("Frozen :", destination)
    print("Source SHA256:", src_hash)
    print("Frozen SHA256:", dst_hash)

    assert src_hash == dst_hash, (
        f"STOP: Hash mismatch while freezing {label}"
    )

    frozen_packages[label] = destination

    print("PASS — byte-for-byte identical")

print("\n" + "=" * 100)
print("✓ R1 and E1–E5/base evidence frozen successfully")
print("✓ Original ZIPs remain untouched")
print("=" * 100)

R2-04 — FREEZE SOURCE PACKAGES

[R1_FINAL]
Source : /content/XAgentic_R1_Phrase_Generalization_FINAL_2026-10-07.zip
Frozen : /content/xagentic_r2_work/00_INPUT_FROZEN/XAgentic_R1_Phrase_Generalization_FINAL_2026-10-07.zip
Source SHA256: bdac27c42a7514700dc878c372c7a2555dd6a1bfcaca9326b1df9900a2e371a4
Frozen SHA256: bdac27c42a7514700dc878c372c7a2555dd6a1bfcaca9326b1df9900a2e371a4
PASS — byte-for-byte identical

[BASE_E1_E5]
Source : /content/XAgentic_Final_Paper_Evidence_2026-10-06 (1).zip
Frozen : /content/xagentic_r2_work/00_INPUT_FROZEN/XAgentic_Final_Paper_Evidence_2026-10-06 (1).zip
Source SHA256: 04a9c98f6cfdf89132fe70f26b7aec5e011805255cd4d202726674d09b8c601a
Frozen SHA256: 04a9c98f6cfdf89132fe70f26b7aec5e011805255cd4d202726674d09b8c601a
PASS — byte-for-byte identical

✓ R1 and E1–E5/base evidence frozen successfully
✓ Original ZIPs remain untouched


In [76]:
# ============================================================
# R2-05 — INSPECT SOURCE PACKAGE STRUCTURE
# ============================================================

import zipfile
from collections import Counter

print("=" * 110)
print("R2-05 — ZIP STRUCTURE INSPECTION")
print("=" * 110)

for label, zip_path in frozen_packages.items():

    print("\n")
    print("#" * 110)
    print(label)
    print("#" * 110)

    with zipfile.ZipFile(zip_path, "r") as z:
        members = [
            x for x in z.infolist()
            if not x.is_dir()
        ]

        print("ZIP:", zip_path.name)
        print("Files:", len(members))

        suffixes = Counter(
            Path(x.filename).suffix.lower() or "<none>"
            for x in members
        )

        print("\nFile types:")
        for suffix, count in suffixes.most_common():
            print(f"  {suffix:12s} {count:5d}")

        print("\nPotentially relevant files:")

        keywords = [
            "feasib",
            "valid",
            "constraint",
            "candidate",
            "policy",
            "slice",
            "intent",
            "kpi",
            "admission",
            "decision",
            "runner",
            "experiment",
            "e1",
            "e2",
            "e3",
            "e4",
            "e5",
        ]

        relevant = []

        for member in members:
            lower = member.filename.lower()

            if any(k in lower for k in keywords):
                relevant.append(member.filename)

        for name in relevant[:250]:
            print(" ", name)

        if len(relevant) > 250:
            print(
                f"\n... {len(relevant)-250} additional "
                "potentially relevant files not displayed"
            )

        print("\nRelevant-file count:", len(relevant))

print("\n✓ Package structure inspection complete")

R2-05 — ZIP STRUCTURE INSPECTION


##############################################################################################################
R1_FINAL
##############################################################################################################
ZIP: XAgentic_R1_Phrase_Generalization_FINAL_2026-10-07.zip
Files: 221

File types:
  .csv           193
  .json           26
  .md              1
  .txt             1

Potentially relevant files:
  R1_phrase_generalization/R1_EXPERIMENT_COMPLETE.json
  R1_phrase_generalization/data/final_conditions/L3_generic_kpi/seed_42.csv
  R1_phrase_generalization/data/final_conditions/L3_generic_kpi/seed_45.csv
  R1_phrase_generalization/data/final_conditions/L3_generic_kpi/seed_46.csv
  R1_phrase_generalization/data/final_conditions/L3_generic_kpi/seed_43.csv
  R1_phrase_generalization/data/final_conditions/L3_generic_kpi/seed_44.csv
  R1_phrase_generalization/data/final_conditions/L4_kpi_only/seed_42.csv
  R1_phrase_generalization/da

In [77]:
# ============================================================
# R2-06 — EXTRACT BASE SOURCE PACKAGE
# ============================================================

import zipfile
from pathlib import Path

ROOT = Path("/content/xagentic_r2_work")
INPUT = ROOT / "00_INPUT_FROZEN"
SOURCE = ROOT / "01_SOURCE"

BASE_FROZEN = (
    INPUT / "XAgentic_Final_Paper_Evidence_2026-10-06 (1).zip"
)

EXTRACT_DIR = SOURCE / "BASE_E1_E5_EXTRACTED"

print("=" * 100)
print("R2-06 — EXTRACT BASE E1–E5 PACKAGE")
print("=" * 100)

assert BASE_FROZEN.exists(), f"STOP: Missing frozen package: {BASE_FROZEN}"

# Safety
if EXTRACT_DIR.exists() and any(EXTRACT_DIR.rglob("*")):
    raise RuntimeError(
        f"STOP: Extraction directory already contains files:\n{EXTRACT_DIR}"
    )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BASE_FROZEN, "r") as z:
    z.extractall(EXTRACT_DIR)

all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]
py_files = [p for p in all_files if p.suffix.lower() == ".py"]

print("Extraction directory:")
print(EXTRACT_DIR)

print("\nTotal files extracted:", len(all_files))
print("Python files found    :", len(py_files))

print("\nPython source files:")
for i, p in enumerate(sorted(py_files), 1):
    print(f"{i:02d}. {p.relative_to(EXTRACT_DIR)}")

assert len(py_files) == 10, (
    f"STOP: Expected 10 Python files based on ZIP audit, "
    f"found {len(py_files)}"
)

print("\n✓ Base package extracted")
print("✓ Expected 10 Python source files located")
print("✓ No source modification performed")

R2-06 — EXTRACT BASE E1–E5 PACKAGE
Extraction directory:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED

Total files extracted: 64
Python files found    : 10

Python source files:
01. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/agents.py
02. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp1_posthoc.py
03. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp2_inhoc.py
04. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp3_semantic.py
05. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp4_combined.py
06. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp5_agentic.py
07. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/run_all.py
08. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/run_seeds.py
09. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_common.py
10. XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_slice.py

✓ Base package extracted
✓ Expected 10 Python source files

In [78]:
# ============================================================
# R2-07 — FREEZE ORIGINAL PYTHON SOURCE FINGERPRINTS
# ============================================================

import hashlib
import pandas as pd
from pathlib import Path

print("=" * 110)
print("R2-07 — ORIGINAL SOURCE FINGERPRINTS")
print("=" * 110)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

rows = []

for p in sorted(py_files):
    rows.append({
        "relative_path": str(p.relative_to(EXTRACT_DIR)),
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

source_manifest = pd.DataFrame(rows)

print(source_manifest.to_string(index=False))

manifest_csv = (
    ROOT /
    "02_R2_AUDIT" /
    "R2_original_source_sha256.csv"
)

source_manifest.to_csv(manifest_csv, index=False)

print("\nManifest saved:")
print(manifest_csv)

print("\n✓ Original Python implementation fingerprinted")
print("✓ These hashes define the pre-R2 baseline")

R2-07 — ORIGINAL SOURCE FINGERPRINTS
                                                             relative_path  size_bytes                                                           sha256
         XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/agents.py       23800 073a9281316ab7be44cbc65e73cf73efdef631400265eb11cee722795039928b
   XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp1_posthoc.py       11817 fc5acf7194e877a5656976043ea13c931a3198b65c0a574affad888a10422c4a
     XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp2_inhoc.py       16222 34b0bec18e401be6417827a1476e674fa625e10f18f12c806383d27b2a776827
  XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp3_semantic.py       12988 7a3366df283101259254c871df9035ed5333a06b80211d4a6d4f47d5bf95f294
  XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/exp4_combined.py        9012 33ce35f3e24530ee6a1c5903a541e429efc70ccc8ab5118fcee1c581bb848597
   XAgentic_Final_Paper_Evidence_2026-10-06/03_SOUR

In [79]:
# ============================================================
# R2-08 — TARGETED FEASIBILITY / VALIDITY SOURCE SCAN
# ============================================================

import pandas as pd

print("=" * 110)
print("R2-08 — FEASIBILITY / VALIDITY SOURCE DISCOVERY")
print("=" * 110)

SEARCH_TERMS = [
    "feasib",
    "valid",
    "candidate",
    "constraint",
    "latency",
    "reliability",
    "bandwidth",
    "throughput",
    "mobility",
    "slice",
    "capacity",
    "density",
    "kpi",
    "admission",
    "policy",
    "fallback",
    "all_slices",
    "escalat",
]

matches = []

for path in sorted(py_files):

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for lineno, line in enumerate(lines, start=1):

        lower = line.lower()

        terms_found = [
            term for term in SEARCH_TERMS
            if term in lower
        ]

        if terms_found:
            matches.append({
                "file": str(path.relative_to(EXTRACT_DIR)),
                "line": lineno,
                "terms": ",".join(terms_found),
                "code": line.strip(),
            })

match_df = pd.DataFrame(matches)

print("Total matching source lines:", len(match_df))

if len(match_df):
    print(
        match_df[
            ["file", "line", "terms", "code"]
        ].to_string(
            index=False,
            max_colwidth=140
        )
    )

output = (
    ROOT /
    "02_R2_AUDIT" /
    "R2_feasibility_source_scan.csv"
)

match_df.to_csv(output, index=False)

print("\nSaved:")
print(output)

print("\n✓ Feasibility/validity source scan complete")

R2-08 — FEASIBILITY / VALIDITY SOURCE DISCOVERY
Total matching source lines: 448
                                                                      file  line                               terms                                                                                       code
         XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/agents.py     2                               valid                 LLM-backed Intent and Validation agents (Together API, OpenAI-compatible).
         XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/agents.py     6                           candidate                   re-do its work; every other stage (candidate screening, Selection Agent,
         XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/agents.py     7                              feasib                    Explanation Agent, feasibility contract) is byte-identical to E1. E5 is
         XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/agents.py    17       

In [80]:
# ============================================================
# R2-09 — INSPECT xagentic_slice.py
# ============================================================

print("=" * 110)
print("R2-09 — xagentic_slice.py INSPECTION")
print("=" * 110)

slice_matches = [
    p for p in py_files
    if p.name == "xagentic_slice.py"
]

assert len(slice_matches) == 1, (
    f"STOP: Expected exactly one xagentic_slice.py; "
    f"found {len(slice_matches)}"
)

SLICE_SOURCE = slice_matches[0]

print("Source:")
print(SLICE_SOURCE)

print("\nSHA256:")
print(sha256_file(SLICE_SOURCE))

lines = SLICE_SOURCE.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

print("\nTotal lines:", len(lines))

print("\n" + "=" * 110)
print("FUNCTION / CLASS DEFINITIONS")
print("=" * 110)

definitions = []

for lineno, line in enumerate(lines, 1):

    stripped = line.strip()

    if (
        stripped.startswith("def ")
        or stripped.startswith("class ")
        or stripped.startswith("async def ")
    ):
        definitions.append((lineno, stripped))
        print(f"{lineno:5d}: {stripped}")

print("\nDefinitions found:", len(definitions))

R2-09 — xagentic_slice.py INSPECTION
Source:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_slice.py

SHA256:
7bf2f1190a2928f273578425f194fc3272385d4baaa9044bff771d0fd748ae05

Total lines: 623

FUNCTION / CLASS DEFINITIONS
  123: def _loguniform(lo, hi, rng):
  127: def _fmt(x):
  137: def synth_corpus(n=3000, p_amb=0.30, seed=SEED):
  239: class Intent:
  250: def parse_intent(text: str) -> Intent:
  292: def featurise(df: pd.DataFrame):
  348: def feasible_set(row):
  365: def contract_valid(pred_label, row):
  373: def rule_based(row):
  391: def main():

Definitions found: 10


In [81]:
# ============================================================
# R2-10 — DISPLAY FEASIBILITY-RELATED CODE WITH CONTEXT
# ============================================================

print("=" * 110)
print("R2-10 — FEASIBILITY CODE CONTEXT")
print("=" * 110)

FOCUS_TERMS = [
    "feasib",
    "candidate",
    "constraint",
    "latency",
    "reliability",
    "bandwidth",
    "throughput",
    "mobility",
    "fallback",
    "valid",
]

hit_lines = []

for i, line in enumerate(lines, start=1):
    lower = line.lower()

    if any(term in lower for term in FOCUS_TERMS):
        hit_lines.append(i)

# Merge nearby hits into larger context blocks
blocks = []

for hit in hit_lines:

    start = max(1, hit - 5)
    end = min(len(lines), hit + 8)

    if not blocks or start > blocks[-1][1] + 1:
        blocks.append([start, end])
    else:
        blocks[-1][1] = max(
            blocks[-1][1],
            end
        )

print("Context blocks found:", len(blocks))

for block_num, (start, end) in enumerate(blocks, 1):

    print("\n" + "-" * 110)
    print(
        f"BLOCK {block_num} — "
        f"LINES {start}–{end}"
    )
    print("-" * 110)

    for lineno in range(start, end + 1):
        print(
            f"{lineno:5d} | "
            f"{lines[lineno - 1]}"
        )

print("\n✓ Context inspection complete")

R2-10 — FEASIBILITY CODE CONTEXT
Context blocks found: 10

--------------------------------------------------------------------------------------------------------------
BLOCK 1 — LINES 3–19
--------------------------------------------------------------------------------------------------------------
    3 | Proof-of-concept experiment pipeline (single comparative experiment).
    4 | 
    5 | Stages
    6 |   S1  Synthetic intent + KPI corpus grounded in 3GPP TS 22.261 / TS 23.501 slice KPIs
    7 |   S2  Intent Agent        : NL intent -> structured intent vector (slot filling)
    8 |   S3  Candidate Agent     : SST/feasibility screening against a slice catalogue
    9 |   S4  Selection Agent     : supervised slice classifier (5 models + rule baseline)
   10 |   S5  Validation Agent    : safety-contract check (3GPP KPI envelopes)
   11 |   S6  Explanation Agent   : SHAP global + local attributions, fidelity + latency
   12 | 
   13 | Outputs: results/*.json, results/*.csv, results/*

In [82]:
# ============================================================
# R2-11 — RECORD BASELINE FEASIBILITY DESIGN
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

AUDIT = ROOT / "02_R2_AUDIT"

baseline_design = {
    "experiment": "R2 Strict Feasibility / Validity Audit",
    "source_file": str(SLICE_SOURCE.relative_to(EXTRACT_DIR)),
    "source_sha256": sha256_file(SLICE_SOURCE),

    "represented_kpis": [
        "latency_ms",
        "throughput_mbps",
        "reliability_pct",
        "density_per_km2",
        "mobility_kmh",
    ],

    "legacy_feasibility_checks": [
        "latency_ms",
        "throughput_mbps",
        "reliability_pct",
        "density_per_km2",
    ],

    "legacy_missing_check": "mobility_kmh",

    "legacy_empty_set_behavior":
        "return all catalogue entries when no candidate remains",

    "legacy_contract":
        "pred_label in feasible_set(row)",

    "audit_principle":
        "R2 does not modify frozen E1-E5 implementation; "
        "strict feasibility is evaluated independently.",

    "created_utc": datetime.now(timezone.utc).isoformat(),
}

out = AUDIT / "R2_baseline_feasibility_design.json"

with open(out, "w", encoding="utf-8") as f:
    json.dump(baseline_design, f, indent=2)

print("=" * 100)
print("R2-11 — BASELINE FEASIBILITY DESIGN")
print("=" * 100)

print(json.dumps(baseline_design, indent=2))

print("\nSaved:")
print(out)

print("\n✓ Frozen baseline behavior documented")

R2-11 — BASELINE FEASIBILITY DESIGN
{
  "experiment": "R2 Strict Feasibility / Validity Audit",
  "source_file": "XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_slice.py",
  "source_sha256": "7bf2f1190a2928f273578425f194fc3272385d4baaa9044bff771d0fd748ae05",
  "represented_kpis": [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh"
  ],
  "legacy_feasibility_checks": [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2"
  ],
  "legacy_missing_check": "mobility_kmh",
  "legacy_empty_set_behavior": "return all catalogue entries when no candidate remains",
  "legacy_contract": "pred_label in feasible_set(row)",
  "audit_principle": "R2 does not modify frozen E1-E5 implementation; strict feasibility is evaluated independently.",
  "created_utc": "2026-10-07T08:13:09.018353+00:00"
}

Saved:
/content/xagentic_r2_work/02_R2_AUDIT/R2_baseline_feasibility_design.json

✓ Frozen baseline beh

In [83]:
# ============================================================
# R2-12 — LOCATE AND DISPLAY FROZEN SLICE CATALOGUE
# ============================================================

print("=" * 110)
print("R2-12 — FROZEN SLICE CATALOGUE")
print("=" * 110)

catalogue_hits = []

for lineno, line in enumerate(lines, 1):
    if "CATALOGUE" in line:
        catalogue_hits.append(lineno)

print("CATALOGUE references:", catalogue_hits)

# Display context around the first occurrence(s)
shown = set()

for hit in catalogue_hits[:5]:

    start = max(1, hit - 10)
    end = min(len(lines), hit + 35)

    key = (start, end)

    if key in shown:
        continue

    shown.add(key)

    print("\n" + "-" * 110)
    print(f"LINES {start}–{end}")
    print("-" * 110)

    for n in range(start, end + 1):
        print(f"{n:5d} | {lines[n-1]}")

print("\n✓ Catalogue inspection complete")

R2-12 — FROZEN SLICE CATALOGUE
CATALOGUE references: [62, 151, 161, 350, 362]

--------------------------------------------------------------------------------------------------------------
LINES 52–97
--------------------------------------------------------------------------------------------------------------
   52 | os.makedirs(OUT, exist_ok=True)
   53 | 
   54 | SLICES = ["eMBB", "URLLC", "mMTC"]
   55 | SLICE_ID = {s: i for i, s in enumerate(SLICES)}
   56 | 
   57 | # ----------------------------------------------------------------------------
   58 | # S1. Slice catalogue + synthetic intent/KPI corpus
   59 | # ----------------------------------------------------------------------------
   60 | 
   61 | # 3GPP-informed KPI envelopes per standardised slice/service type (SST).
   62 | CATALOGUE = {
   63 |     "eMBB":  dict(lat=(10.0, 150.0), thr=(25.0, 1000.0), rel=(99.0, 99.9),
   64 |                   den=(1e2, 5e4), mob=(0.0, 250.0), sst=1),
   65 |     "URLLC": dict(lat=(0.

In [84]:
# ============================================================
# R2-13 — IMPORT FROZEN IMPLEMENTATION
# ============================================================

import sys
import importlib.util

print("=" * 100)
print("R2-13 — IMPORT FROZEN IMPLEMENTATION")
print("=" * 100)

source_dir = SLICE_SOURCE.parent

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

spec = importlib.util.spec_from_file_location(
    "xagentic_slice_frozen",
    SLICE_SOURCE
)

frozen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(frozen)

print("Imported:")
print(SLICE_SOURCE)

print("\nSlices:")
print(frozen.SLICES)

print("\nCatalogue:")
for slice_name, capability in frozen.CATALOGUE.items():
    print(f"\n{slice_name}")
    print(capability)

print("\nLegacy functions:")
print("feasible_set   :", frozen.feasible_set)
print("contract_valid :", frozen.contract_valid)

assert sha256_file(SLICE_SOURCE) == \
    "7bf2f1190a2928f273578425f194fc3272385d4baaa9044bff771d0fd748ae05"

print("\n✓ Frozen implementation imported")
print("✓ SHA-256 baseline unchanged")

R2-13 — IMPORT FROZEN IMPLEMENTATION


AttributeError: 'NoneType' object has no attribute '__dict__'

In [85]:
# ============================================================
# R2-13B — IMPORT FROZEN IMPLEMENTATION
# Python 3.13-safe dynamic import
# ============================================================

import sys
import importlib.util

print("=" * 100)
print("R2-13B — IMPORT FROZEN IMPLEMENTATION")
print("=" * 100)

source_dir = SLICE_SOURCE.parent

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

MODULE_NAME = "xagentic_slice_frozen"

# Remove any incomplete module left by the failed attempt
sys.modules.pop(MODULE_NAME, None)

spec = importlib.util.spec_from_file_location(
    MODULE_NAME,
    SLICE_SOURCE
)

assert spec is not None
assert spec.loader is not None

frozen = importlib.util.module_from_spec(spec)

# IMPORTANT for Python 3.13 + @dataclass
sys.modules[MODULE_NAME] = frozen

try:
    spec.loader.exec_module(frozen)
except Exception:
    # Avoid retaining a half-imported module
    sys.modules.pop(MODULE_NAME, None)
    raise

print("Imported:")
print(SLICE_SOURCE)

print("\nSHA256:")
current_hash = sha256_file(SLICE_SOURCE)
print(current_hash)

EXPECTED_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

assert current_hash == EXPECTED_HASH, (
    "STOP: frozen source SHA-256 changed"
)

print("\nSlices:")
print(frozen.SLICES)

print("\nCatalogue:")
for slice_name, capability in frozen.CATALOGUE.items():
    print(f"\n{slice_name}:")
    print(capability)

print("\nLegacy functions:")
print("feasible_set   :", frozen.feasible_set)
print("contract_valid :", frozen.contract_valid)

print("\n✓ Frozen implementation imported successfully")
print("✓ Python 3.13 dataclass import issue resolved")
print("✓ SHA-256 baseline unchanged")
print("✓ No source modification performed")

R2-13B — IMPORT FROZEN IMPLEMENTATION
Imported:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_slice.py

SHA256:
7bf2f1190a2928f273578425f194fc3272385d4baaa9044bff771d0fd748ae05

Slices:
['eMBB', 'URLLC', 'mMTC']

Catalogue:

eMBB:
{'lat': (10.0, 150.0), 'thr': (25.0, 1000.0), 'rel': (99.0, 99.9), 'den': (100.0, 50000.0), 'mob': (0.0, 250.0), 'sst': 1}

URLLC:
{'lat': (0.5, 10.0), 'thr': (0.1, 100.0), 'rel': (99.99, 99.9999), 'den': (100.0, 10000.0), 'mob': (0.0, 500.0), 'sst': 2}

mMTC:
{'lat': (50.0, 10000.0), 'thr': (0.001, 2.0), 'rel': (95.0, 99.9), 'den': (10000.0, 1000000.0), 'mob': (0.0, 30.0), 'sst': 3}

Legacy functions:
feasible_set   : <function feasible_set at 0x78fd3a566fc0>
contract_valid : <function contract_valid at 0x78fd3a564180>

✓ Frozen implementation imported successfully
✓ Python 3.13 dataclass import issue resolved
✓ SHA-256 baseline unchanged
✓ No source modification performed


In [86]:
# ============================================================
# R2-14 — LEGACY KPI COVERAGE AUDIT
# ============================================================

import inspect
import pandas as pd

print("=" * 100)
print("R2-14 — LEGACY KPI COVERAGE AUDIT")
print("=" * 100)

legacy_source = inspect.getsource(frozen.feasible_set)

print("\nLegacy feasible_set():")
print("-" * 80)
print(legacy_source)

represented = [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
]

coverage_rows = []

for kpi in represented:

    used = kpi in legacy_source

    coverage_rows.append({
        "kpi": kpi,
        "represented_by_intent_model": True,
        "checked_by_legacy_feasible_set": used,
        "audit_status": (
            "COVERED"
            if used
            else "MISSING_FROM_LEGACY_FEASIBILITY"
        ),
    })

coverage = pd.DataFrame(coverage_rows)

print("\nKPI COVERAGE")
print("-" * 100)
print(coverage.to_string(index=False))

coverage_path = (
    ROOT /
    "02_R2_AUDIT" /
    "R2_legacy_kpi_coverage.csv"
)

coverage.to_csv(
    coverage_path,
    index=False
)

# ------------------------------------------------------------
# Assertions: fail loudly if our source inspection is wrong
# ------------------------------------------------------------

assert len(coverage) == 5

assert coverage[
    "represented_by_intent_model"
].all()

for expected_kpi in [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
]:
    assert coverage.loc[
        coverage["kpi"] == expected_kpi,
        "checked_by_legacy_feasible_set"
    ].iloc[0] == True

assert coverage.loc[
    coverage["kpi"] == "mobility_kmh",
    "checked_by_legacy_feasible_set"
].iloc[0] == False

print("\nSaved:")
print(coverage_path)

print("\nSUMMARY")
print("-" * 100)

n_represented = int(
    coverage["represented_by_intent_model"].sum()
)

n_checked = int(
    coverage["checked_by_legacy_feasible_set"].sum()
)

n_missing = n_represented - n_checked

print(f"Represented KPI dimensions : {n_represented}")
print(f"Legacy feasibility checks  : {n_checked}")
print(f"Missing dimensions         : {n_missing}")
print("Missing KPI                : mobility_kmh")

print("\n✓ KPI coverage audit complete")
print("✓ Mobility omission independently confirmed")

R2-14 — LEGACY KPI COVERAGE AUDIT

Legacy feasible_set():
--------------------------------------------------------------------------------
def feasible_set(row):
    out = []
    for s, c in CATALOGUE.items():
        ok = True
        if np.isfinite(row["latency_ms"]) and row["latency_ms"] < c["lat"][0] * 0.5:
            ok = False
        if np.isfinite(row["throughput_mbps"]) and row["throughput_mbps"] > c["thr"][1] * 1.5:
            ok = False
        if np.isfinite(row["reliability_pct"]) and row["reliability_pct"] > c["rel"][1] + 1e-6:
            ok = False
        if np.isfinite(row["density_per_km2"]) and row["density_per_km2"] > c["den"][1] * 2.0:
            ok = False
        if ok:
            out.append(s)
    return out or list(CATALOGUE)


KPI COVERAGE
----------------------------------------------------------------------------------------------------
            kpi  represented_by_intent_model  checked_by_legacy_feasible_set                    audit_status
     late

In [87]:
# ============================================================
# R2-15 — INDEPENDENT STRICT FEASIBILITY ORACLE
# ============================================================

import numpy as np
import pandas as pd
import json
import inspect
import hashlib
from pathlib import Path

print("=" * 110)
print("R2-15 — INDEPENDENT STRICT FEASIBILITY ORACLE")
print("=" * 110)

AUDIT_DIR = ROOT / "02_R2_AUDIT"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# IMPORTANT:
# This is an R2-only independent oracle.
# frozen.feasible_set() and xagentic_slice.py are NOT modified.
#
# Semantics of an operator requirement:
#
# latency_ms       <= slice maximum supported latency
# throughput_mbps  <= slice maximum supported throughput
# reliability_pct  <= slice maximum supported reliability
# density_per_km2  <= slice maximum supported density
# mobility_kmh     <= slice maximum supported mobility
#
# Why use catalogue maxima?
# The request expresses required capability:
#   "latency under X"
#   "at least X Mbps"
#   "X% reliability"
#   "X devices/km2"
#   "mobility up to X km/h"
#
# Therefore a slice is feasible when its capability can satisfy
# the stated requirement.
#
# Missing / NaN KPI => no constraint.
# Empty feasible set => [].
# There is NO fallback to all slices.
# ------------------------------------------------------------------

KPI_RULES = {
    "latency_ms": {
        "catalogue_key": "lat",
        "comparison": "request<=catalogue_max",
    },
    "throughput_mbps": {
        "catalogue_key": "thr",
        "comparison": "request<=catalogue_max",
    },
    "reliability_pct": {
        "catalogue_key": "rel",
        "comparison": "request<=catalogue_max",
    },
    "density_per_km2": {
        "catalogue_key": "den",
        "comparison": "request<=catalogue_max",
    },
    "mobility_kmh": {
        "catalogue_key": "mob",
        "comparison": "request<=catalogue_max",
    },
}


def _finite_value(row, key):
    """Return finite float value or None for a missing constraint."""
    try:
        value = float(row[key])
    except (KeyError, TypeError, ValueError):
        return None

    return value if np.isfinite(value) else None


def strict_slice_audit(slice_name, row):
    """
    Independently audit one slice against every stated KPI.

    Returns:
        feasible: bool
        violations: list[str]
        checks: dict
    """

    c = frozen.CATALOGUE[slice_name]

    violations = []
    checks = {}

    for kpi, rule in KPI_RULES.items():

        request_value = _finite_value(row, kpi)
        catalogue_key = rule["catalogue_key"]

        lower, upper = c[catalogue_key]

        if request_value is None:
            checks[kpi] = {
                "stated": False,
                "request": None,
                "catalogue_min": float(lower),
                "catalogue_max": float(upper),
                "pass": None,
            }
            continue

        # Capability requirement:
        # requested value must not exceed slice capability maximum.
        passed = request_value <= (float(upper) + 1e-12)

        checks[kpi] = {
            "stated": True,
            "request": float(request_value),
            "catalogue_min": float(lower),
            "catalogue_max": float(upper),
            "pass": bool(passed),
        }

        if not passed:
            violations.append(kpi)

    return {
        "slice": slice_name,
        "feasible": len(violations) == 0,
        "violations": violations,
        "checks": checks,
    }


def strict_feasible_set(row):
    """
    Return only slices satisfying ALL stated KPI constraints.

    Critical R2 behavior:
        no feasible slices -> []
    """

    feasible = []

    for slice_name in frozen.SLICES:

        audit = strict_slice_audit(
            slice_name,
            row
        )

        if audit["feasible"]:
            feasible.append(slice_name)

    return feasible


def strict_contract_valid(pred_label, row):
    """
    Strict validity of a selected slice.
    """

    return pred_label in strict_feasible_set(row)


# ------------------------------------------------------------------
# Save oracle source itself as R2 evidence
# ------------------------------------------------------------------

oracle_source = (
    inspect.getsource(_finite_value)
    + "\n\n"
    + inspect.getsource(strict_slice_audit)
    + "\n\n"
    + inspect.getsource(strict_feasible_set)
    + "\n\n"
    + inspect.getsource(strict_contract_valid)
)

oracle_path = (
    AUDIT_DIR /
    "R2_strict_feasibility_oracle.py"
)

oracle_path.write_text(
    oracle_source,
    encoding="utf-8"
)

oracle_sha256 = hashlib.sha256(
    oracle_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------------
# Verify frozen baseline is still unchanged
# ------------------------------------------------------------------

EXPECTED_FROZEN_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

current_frozen_hash = sha256_file(
    SLICE_SOURCE
)

assert current_frozen_hash == EXPECTED_FROZEN_HASH


# ------------------------------------------------------------------
# Structural assertions
# ------------------------------------------------------------------

assert set(KPI_RULES) == {
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
}

# Empty impossible request must remain empty.
impossible = pd.Series({
    "latency_ms": 1.0,
    "throughput_mbps": 5000.0,
    "reliability_pct": 99.99999,
    "density_per_km2": 2_000_000.0,
    "mobility_kmh": 1000.0,
})

strict_impossible = strict_feasible_set(
    impossible
)

legacy_impossible = frozen.feasible_set(
    impossible
)

assert strict_impossible == [], (
    "STOP: strict oracle must preserve empty feasible set"
)

print("\nKPI rules:")
for k, v in KPI_RULES.items():
    print(
        f"  {k:20s} -> "
        f"{v['catalogue_key']:4s} | "
        f"{v['comparison']}"
    )

print("\nImpossible-case sanity test:")
print("Legacy feasible set :", legacy_impossible)
print("Strict feasible set :", strict_impossible)

print("\nOracle saved:")
print(oracle_path)

print("\nOracle SHA256:")
print(oracle_sha256)

print("\nFrozen source SHA256:")
print(current_frozen_hash)

print("\n✓ Strict oracle created")
print("✓ All five represented KPI dimensions audited")
print("✓ Missing KPI values treated as unconstrained")
print("✓ Empty feasible set preserved as []")
print("✓ Frozen E1–E5 source unchanged")

R2-15 — INDEPENDENT STRICT FEASIBILITY ORACLE

KPI rules:
  latency_ms           -> lat  | request<=catalogue_max
  throughput_mbps      -> thr  | request<=catalogue_max
  reliability_pct      -> rel  | request<=catalogue_max
  density_per_km2      -> den  | request<=catalogue_max
  mobility_kmh         -> mob  | request<=catalogue_max

Impossible-case sanity test:
Legacy feasible set : ['eMBB', 'URLLC', 'mMTC']
Strict feasible set : []

Oracle saved:
/content/xagentic_r2_work/02_R2_AUDIT/R2_strict_feasibility_oracle.py

Oracle SHA256:
5102ab0bdea2e732f99b34b9c8a371730816a3f63686c51a2cc8a55584003ec6

Frozen source SHA256:
7bf2f1190a2928f273578425f194fc3272385d4baaa9044bff771d0fd748ae05

✓ Strict oracle created
✓ All five represented KPI dimensions audited
✓ Missing KPI values treated as unconstrained
✓ Empty feasible set preserved as []
✓ Frozen E1–E5 source unchanged


In [88]:
# ============================================================
# R2-16 — CONTROLLED STRICT-FEASIBILITY VALIDATION SUITE
# ============================================================

import numpy as np
import pandas as pd
import json

print("=" * 120)
print("R2-16 — CONTROLLED STRICT-FEASIBILITY VALIDATION SUITE")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_RAW.mkdir(parents=True, exist_ok=True)

NAN = np.nan

# ------------------------------------------------------------------
# Controlled cases
#
# These are deliberately small and interpretable.
# They test:
#   1. normal feasible cases
#   2. exact capability boundaries
#   3. each KPI independently
#   4. mobility omission
#   5. multiple simultaneous violations
#   6. no-feasible-slice cases
#   7. missing KPI handling
#
# expected_strict_set is determined directly from the frozen catalogue.
# ------------------------------------------------------------------

cases = [

    # --------------------------------------------------------------
    # Basic feasible cases
    # --------------------------------------------------------------
    {
        "case_id": "F01",
        "category": "feasible",
        "description": "Moderate eMBB-like requirement",
        "latency_ms": 100.0,
        "throughput_mbps": 100.0,
        "reliability_pct": 99.5,
        "density_per_km2": 1000.0,
        "mobility_kmh": 100.0,
        "expected_strict_set": ["eMBB"],
    },

    {
        "case_id": "F02",
        "category": "feasible",
        "description": "Strict URLLC-like requirement",
        "latency_ms": 5.0,
        "throughput_mbps": 50.0,
        "reliability_pct": 99.995,
        "density_per_km2": 5000.0,
        "mobility_kmh": 300.0,
        "expected_strict_set": ["URLLC"],
    },

    {
        "case_id": "F03",
        "category": "feasible",
        "description": "mMTC-like requirement",
        "latency_ms": 1000.0,
        "throughput_mbps": 1.0,
        "reliability_pct": 98.0,
        "density_per_km2": 500000.0,
        "mobility_kmh": 20.0,
        "expected_strict_set": ["mMTC"],
    },

    # --------------------------------------------------------------
    # Exact catalogue upper boundaries
    # --------------------------------------------------------------
    {
        "case_id": "B01",
        "category": "boundary",
        "description": "eMBB exact upper capability boundaries",
        "latency_ms": 150.0,
        "throughput_mbps": 1000.0,
        "reliability_pct": 99.9,
        "density_per_km2": 50000.0,
        "mobility_kmh": 250.0,
        "expected_strict_set": ["eMBB"],
    },

    {
        "case_id": "B02",
        "category": "boundary",
        "description": "URLLC exact upper capability boundaries",
        "latency_ms": 10.0,
        "throughput_mbps": 100.0,
        "reliability_pct": 99.9999,
        "density_per_km2": 10000.0,
        "mobility_kmh": 500.0,
        "expected_strict_set": ["URLLC"],
    },

    {
        "case_id": "B03",
        "category": "boundary",
        "description": "mMTC exact upper capability boundaries",
        "latency_ms": 10000.0,
        "throughput_mbps": 2.0,
        "reliability_pct": 99.9,
        "density_per_km2": 1000000.0,
        "mobility_kmh": 30.0,
        "expected_strict_set": ["mMTC"],
    },

    # --------------------------------------------------------------
    # Mobility-only tests
    # Other KPIs are missing so mobility is isolated.
    # --------------------------------------------------------------
    {
        "case_id": "M01",
        "category": "mobility_only",
        "description": "Mobility 20 km/h: all slices support",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": 20.0,
        "expected_strict_set": ["eMBB", "URLLC", "mMTC"],
    },

    {
        "case_id": "M02",
        "category": "mobility_only",
        "description": "Mobility 100 km/h excludes mMTC",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": 100.0,
        "expected_strict_set": ["eMBB", "URLLC"],
    },

    {
        "case_id": "M03",
        "category": "mobility_only",
        "description": "Mobility 300 km/h leaves only URLLC",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": 300.0,
        "expected_strict_set": ["URLLC"],
    },

    {
        "case_id": "M04",
        "category": "mobility_only",
        "description": "Mobility above every catalogue capability",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": 600.0,
        "expected_strict_set": [],
    },

    # --------------------------------------------------------------
    # Individual KPI impossibility
    # --------------------------------------------------------------
    {
        "case_id": "I01",
        "category": "single_kpi_infeasible",
        "description": "Throughput exceeds every slice capability",
        "latency_ms": NAN,
        "throughput_mbps": 1500.0,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": NAN,
        "expected_strict_set": [],
    },

    {
        "case_id": "I02",
        "category": "single_kpi_infeasible",
        "description": "Reliability exceeds every slice capability",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": 99.99999,
        "density_per_km2": NAN,
        "mobility_kmh": NAN,
        "expected_strict_set": [],
    },

    {
        "case_id": "I03",
        "category": "single_kpi_infeasible",
        "description": "Density exceeds every slice capability",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": NAN,
        "density_per_km2": 1500000.0,
        "mobility_kmh": NAN,
        "expected_strict_set": [],
    },

    # --------------------------------------------------------------
    # Multi-KPI contradiction / impossible cases
    # --------------------------------------------------------------
    {
        "case_id": "X01",
        "category": "multi_kpi_infeasible",
        "description": "Extreme requirements violate several capabilities",
        "latency_ms": 1.0,
        "throughput_mbps": 5000.0,
        "reliability_pct": 99.99999,
        "density_per_km2": 2000000.0,
        "mobility_kmh": 1000.0,
        "expected_strict_set": [],
    },

    {
        "case_id": "X02",
        "category": "cross_kpi_conflict",
        "description": "High throughput plus very high reliability has no matching slice",
        "latency_ms": NAN,
        "throughput_mbps": 500.0,
        "reliability_pct": 99.99,
        "density_per_km2": NAN,
        "mobility_kmh": NAN,
        "expected_strict_set": [],
    },

    # --------------------------------------------------------------
    # Missing KPI behavior
    # --------------------------------------------------------------
    {
        "case_id": "N01",
        "category": "missing_kpis",
        "description": "No KPI stated: all slices remain possible",
        "latency_ms": NAN,
        "throughput_mbps": NAN,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": NAN,
        "expected_strict_set": ["eMBB", "URLLC", "mMTC"],
    },

    {
        "case_id": "N02",
        "category": "partial_kpis",
        "description": "Only throughput 50 Mbps stated",
        "latency_ms": NAN,
        "throughput_mbps": 50.0,
        "reliability_pct": NAN,
        "density_per_km2": NAN,
        "mobility_kmh": NAN,
        "expected_strict_set": ["eMBB", "URLLC"],
    },
]


# ------------------------------------------------------------------
# Run legacy and strict implementations
# ------------------------------------------------------------------

rows = []

for case in cases:

    row = pd.Series({
        "latency_ms": case["latency_ms"],
        "throughput_mbps": case["throughput_mbps"],
        "reliability_pct": case["reliability_pct"],
        "density_per_km2": case["density_per_km2"],
        "mobility_kmh": case["mobility_kmh"],
    })

    legacy_set = frozen.feasible_set(row)
    strict_set = strict_feasible_set(row)

    expected = case["expected_strict_set"]

    strict_pass = strict_set == expected

    legacy_empty_fallback = (
        strict_set == []
        and legacy_set == list(frozen.CATALOGUE)
    )

    mobility_difference = (
        np.isfinite(case["mobility_kmh"])
        and legacy_set != strict_set
    )

    rows.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "description": case["description"],

        "latency_ms": case["latency_ms"],
        "throughput_mbps": case["throughput_mbps"],
        "reliability_pct": case["reliability_pct"],
        "density_per_km2": case["density_per_km2"],
        "mobility_kmh": case["mobility_kmh"],

        "expected_strict_set": "|".join(expected),
        "strict_set": "|".join(strict_set),
        "legacy_set": "|".join(legacy_set),

        "strict_test_pass": strict_pass,
        "legacy_differs_from_strict": legacy_set != strict_set,
        "legacy_empty_fallback_exposed": legacy_empty_fallback,
        "mobility_difference": mobility_difference,
    })


results = pd.DataFrame(rows)


# ------------------------------------------------------------------
# Save complete evidence
# ------------------------------------------------------------------

out_csv = (
    RESULTS_RAW /
    "R2_controlled_feasibility_cases.csv"
)

results.to_csv(
    out_csv,
    index=False
)


# ------------------------------------------------------------------
# Assertions
# ------------------------------------------------------------------

assert results["strict_test_pass"].all(), (
    "STOP: one or more strict-oracle controlled tests failed"
)

assert (
    results.loc[
        results["case_id"] == "M02",
        "legacy_differs_from_strict"
    ].iloc[0]
)

assert (
    results.loc[
        results["case_id"] == "M03",
        "legacy_differs_from_strict"
    ].iloc[0]
)

assert (
    results.loc[
        results["case_id"] == "M04",
        "legacy_empty_fallback_exposed"
    ].iloc[0]
)

assert (
    results.loc[
        results["case_id"] == "X01",
        "legacy_empty_fallback_exposed"
    ].iloc[0]
)


# ------------------------------------------------------------------
# Print concise results
# ------------------------------------------------------------------

display_cols = [
    "case_id",
    "category",
    "expected_strict_set",
    "strict_set",
    "legacy_set",
    "legacy_differs_from_strict",
    "legacy_empty_fallback_exposed",
]

print("\nCONTROLLED TEST RESULTS")
print("-" * 120)

print(
    results[display_cols].to_string(
        index=False
    )
)

print("\nSUMMARY")
print("-" * 120)

print(
    "Controlled cases                  :",
    len(results)
)

print(
    "Strict oracle tests passed        :",
    int(results["strict_test_pass"].sum()),
    "/",
    len(results)
)

print(
    "Legacy/strict disagreements       :",
    int(results["legacy_differs_from_strict"].sum())
)

print(
    "Empty-fallback defects exposed    :",
    int(results["legacy_empty_fallback_exposed"].sum())
)

print(
    "Mobility-related disagreements    :",
    int(results["mobility_difference"].sum())
)

print("\nSaved:")
print(out_csv)

print("\n✓ Controlled strict-feasibility suite complete")
print("✓ All strict-oracle expectations satisfied")
print("✓ Legacy comparison preserved")

R2-16 — CONTROLLED STRICT-FEASIBILITY VALIDATION SUITE

CONTROLLED TEST RESULTS
------------------------------------------------------------------------------------------------------------------------
case_id              category expected_strict_set      strict_set      legacy_set  legacy_differs_from_strict  legacy_empty_fallback_exposed
    F01              feasible                eMBB            eMBB      eMBB|URLLC                        True                          False
    F02              feasible               URLLC           URLLC           URLLC                       False                          False
    F03              feasible                mMTC            mMTC            mMTC                       False                          False
    B01              boundary                eMBB            eMBB            eMBB                       False                          False
    B02              boundary               URLLC           URLLC           URLLC             

In [89]:
# ============================================================
# R2-17 — LOCATE AND VALIDATE FROZEN EXPERIMENTAL CORPUS
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import os

print("=" * 120)
print("R2-17 — LOCATE AND VALIDATE FROZEN EXPERIMENTAL CORPUS")
print("=" * 120)

# ------------------------------------------------------------
# 1. Search ONLY inside the extracted frozen E1-E5 package
# ------------------------------------------------------------

search_root = (
    ROOT /
    "01_SOURCE" /
    "BASE_E1_E5_EXTRACTED"
)

assert search_root.exists(), (
    f"STOP: frozen extraction root missing: {search_root}"
)

corpus_candidates = sorted(
    p for p in search_root.rglob("corpus.csv")
    if p.is_file()
)

print("\nCorpus candidates found:")
print("-" * 120)

if not corpus_candidates:
    print("NONE")
else:
    for i, p in enumerate(corpus_candidates, 1):
        print(f"{i:2d}. {p}")
        print(f"    size = {p.stat().st_size:,} bytes")


# ------------------------------------------------------------
# 2. Require at least one existing frozen corpus
# ------------------------------------------------------------

assert corpus_candidates, (
    "STOP: no corpus.csv found inside frozen E1-E5 package. "
    "Do NOT regenerate it."
)


# ------------------------------------------------------------
# 3. Inspect every candidate
# ------------------------------------------------------------

def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


required_truth_columns = {
    "id",
    "text",
    "slice",
    "lat_true",
    "thr_true",
    "rel_true",
    "den_true",
    "mob_true",
}

candidate_rows = []

print("\nCANDIDATE INSPECTION")
print("-" * 120)

for p in corpus_candidates:

    df_tmp = pd.read_csv(p)

    sha = file_sha256(p)

    missing = sorted(
        required_truth_columns -
        set(df_tmp.columns)
    )

    candidate_rows.append({
        "path": str(p),
        "rows": len(df_tmp),
        "columns": len(df_tmp.columns),
        "size_bytes": p.stat().st_size,
        "sha256": sha,
        "has_required_truth_columns": len(missing) == 0,
        "missing_required_columns": "|".join(missing),
    })

    print("\nPATH:")
    print(p)

    print("Rows        :", len(df_tmp))
    print("Columns     :", len(df_tmp.columns))
    print("SHA256      :", sha)

    print("Required KPI truth columns:",
          "PASS" if not missing else "FAIL")

    if missing:
        print("Missing:", missing)


candidate_table = pd.DataFrame(
    candidate_rows
)


# ------------------------------------------------------------
# 4. Select valid candidate
#
# Prefer a candidate with:
#   - required columns
#   - 3000 rows (original main experiment size)
#
# Do not silently choose if ambiguity remains.
# ------------------------------------------------------------

valid = candidate_table[
    candidate_table[
        "has_required_truth_columns"
    ]
].copy()

assert len(valid) > 0, (
    "STOP: corpus candidates exist, but none contain "
    "the required frozen KPI truth columns."
)

preferred = valid[
    valid["rows"] == 3000
].copy()

if len(preferred) == 1:

    selected_path = Path(
        preferred.iloc[0]["path"]
    )

elif len(valid) == 1:

    selected_path = Path(
        valid.iloc[0]["path"]
    )

else:

    print("\nVALID CANDIDATES")
    print("-" * 120)

    print(
        valid[
            [
                "path",
                "rows",
                "columns",
                "size_bytes",
                "sha256",
            ]
        ].to_string(index=False)
    )

    raise RuntimeError(
        "STOP: multiple valid frozen corpus candidates remain. "
        "Do not choose automatically."
    )


# ------------------------------------------------------------
# 5. Load selected frozen corpus
# ------------------------------------------------------------

CORPUS_PATH = selected_path

corpus = pd.read_csv(
    CORPUS_PATH
)

CORPUS_SHA256 = file_sha256(
    CORPUS_PATH
)


# ------------------------------------------------------------
# 6. Validate expected structure
# ------------------------------------------------------------

assert required_truth_columns.issubset(
    corpus.columns
)

assert len(corpus) > 0

assert set(
    corpus["slice"].dropna().unique()
).issubset(
    set(frozen.SLICES)
)


# ------------------------------------------------------------
# 7. Report missingness of each generated KPI
# ------------------------------------------------------------

truth_columns = [
    "lat_true",
    "thr_true",
    "rel_true",
    "den_true",
    "mob_true",
]

print("\nSELECTED FROZEN CORPUS")
print("-" * 120)

print("Path:")
print(CORPUS_PATH)

print("\nSHA256:")
print(CORPUS_SHA256)

print("\nShape:")
print(corpus.shape)

print("\nColumns:")
print(list(corpus.columns))

print("\nGround-truth slice counts:")
print(
    corpus["slice"]
    .value_counts()
    .to_string()
)

print("\nKPI truth availability:")
for col in truth_columns:

    present = int(
        corpus[col].notna().sum()
    )

    missing = int(
        corpus[col].isna().sum()
    )

    print(
        f"{col:12s} "
        f"present={present:4d} "
        f"missing={missing:4d}"
    )


# ------------------------------------------------------------
# 8. Save corpus provenance evidence
# ------------------------------------------------------------

provenance_path = (
    AUDIT_DIR /
    "R2_frozen_corpus_provenance.csv"
)

candidate_table.to_csv(
    provenance_path,
    index=False
)

print("\nProvenance saved:")
print(provenance_path)


# ------------------------------------------------------------
# 9. Re-verify source immutability
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

print("\n✓ Frozen corpus located")
print("✓ Frozen corpus provenance recorded")
print("✓ KPI truth columns validated")
print("✓ No corpus regeneration performed")
print("✓ Frozen E1-E5 source unchanged")

R2-17 — LOCATE AND VALIDATE FROZEN EXPERIMENTAL CORPUS

Corpus candidates found:
------------------------------------------------------------------------------------------------------------------------
 1. /content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/02_E5_QWEN37_N100_SEED52/corpus.csv
    size = 307,478 bytes

CANDIDATE INSPECTION
------------------------------------------------------------------------------------------------------------------------

PATH:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/02_E5_QWEN37_N100_SEED52/corpus.csv
Rows        : 1200
Columns     : 12
SHA256      : 5358e25a06c863fcb25b4d906608783d612c0b1411373f140573408555b04b08
Required KPI truth columns: PASS

SELECTED FROZEN CORPUS
------------------------------------------------------------------------------------------------------------------------
Path:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRA

In [90]:
# ============================================================
# R2-18 — DISCOVER FROZEN AUDIT INPUT ARTIFACTS
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

print("=" * 120)
print("R2-18 — DISCOVER FROZEN AUDIT INPUT ARTIFACTS")
print("=" * 120)

BASE_ROOT = (
    ROOT /
    "01_SOURCE" /
    "BASE_E1_E5_EXTRACTED" /
    "XAgentic_Final_Paper_Evidence_2026-10-06"
)

assert BASE_ROOT.exists(), f"STOP: base evidence root missing: {BASE_ROOT}"

# Terms that may identify useful audit evidence.
search_terms = [
    "latency",
    "throughput",
    "reliability",
    "density",
    "mobility",
    "lat_true",
    "thr_true",
    "rel_true",
    "den_true",
    "mob_true",
    "pred",
    "prediction",
    "predicted",
    "label",
    "slice",
    "contract",
    "valid",
    "feasible",
    "escalat",
]

artifact_rows = []

csv_files = sorted(BASE_ROOT.rglob("*.csv"))
json_files = sorted(BASE_ROOT.rglob("*.json"))

print(f"\nCSV files found  : {len(csv_files)}")
print(f"JSON files found : {len(json_files)}")

# ------------------------------------------------------------
# CSV inspection
# ------------------------------------------------------------

print("\nCSV ARTIFACTS WITH POTENTIALLY RELEVANT COLUMNS")
print("-" * 120)

for path in csv_files:

    try:
        df = pd.read_csv(path)
    except Exception as exc:
        artifact_rows.append({
            "type": "CSV",
            "path": str(path),
            "rows": None,
            "columns": None,
            "relevant_fields": "",
            "sha256": file_sha256(path),
            "status": f"READ_ERROR: {exc}",
        })
        continue

    cols = [str(c) for c in df.columns]

    relevant = [
        c for c in cols
        if any(
            term in c.lower()
            for term in search_terms
        )
    ]

    artifact_rows.append({
        "type": "CSV",
        "path": str(path),
        "rows": len(df),
        "columns": len(cols),
        "relevant_fields": "|".join(relevant),
        "sha256": file_sha256(path),
        "status": "OK",
    })

    if relevant:
        print("\nFILE:")
        print(path.relative_to(BASE_ROOT))
        print("Shape :", df.shape)
        print("Relevant columns:")
        print(relevant)

# ------------------------------------------------------------
# JSON inspection — top-level keys only
# ------------------------------------------------------------

print("\nJSON ARTIFACTS WITH POTENTIALLY RELEVANT KEYS")
print("-" * 120)

for path in json_files:

    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            keys = [str(k) for k in obj.keys()]
        elif isinstance(obj, list) and obj and isinstance(obj[0], dict):
            keys = [str(k) for k in obj[0].keys()]
        else:
            keys = []

        relevant = [
            k for k in keys
            if any(
                term in k.lower()
                for term in search_terms
            )
        ]

        artifact_rows.append({
            "type": "JSON",
            "path": str(path),
            "rows": len(obj) if isinstance(obj, list) else None,
            "columns": len(keys),
            "relevant_fields": "|".join(relevant),
            "sha256": file_sha256(path),
            "status": "OK",
        })

        if relevant:
            print("\nFILE:")
            print(path.relative_to(BASE_ROOT))
            print("Relevant keys:")
            print(relevant)

    except Exception as exc:

        artifact_rows.append({
            "type": "JSON",
            "path": str(path),
            "rows": None,
            "columns": None,
            "relevant_fields": "",
            "sha256": file_sha256(path),
            "status": f"READ_ERROR: {exc}",
        })

# ------------------------------------------------------------
# Save complete inventory
# ------------------------------------------------------------

inventory = pd.DataFrame(artifact_rows)

inventory_path = (
    AUDIT_DIR /
    "R2_frozen_result_artifact_inventory.csv"
)

inventory.to_csv(
    inventory_path,
    index=False
)

# ------------------------------------------------------------
# Identify strongest candidate CSVs
# ------------------------------------------------------------

def score_fields(fields):

    s = fields.lower()

    score = 0

    # Parsed KPI slots are particularly valuable.
    for term in [
        "latency_ms",
        "throughput_mbps",
        "reliability_pct",
        "density_per_km2",
        "mobility_kmh",
    ]:
        if term in s:
            score += 5

    # Predictions / selections.
    for term in [
        "pred",
        "prediction",
        "selected",
    ]:
        if term in s:
            score += 3

    # Existing safety result.
    for term in [
        "contract",
        "valid",
        "feasible",
        "escalat",
    ]:
        if term in s:
            score += 2

    return score


csv_inventory = inventory[
    inventory["type"] == "CSV"
].copy()

csv_inventory["audit_score"] = (
    csv_inventory["relevant_fields"]
    .fillna("")
    .map(score_fields)
)

ranked = csv_inventory.sort_values(
    ["audit_score", "rows"],
    ascending=[False, False],
)

print("\nTOP CANDIDATE CSV ARTIFACTS")
print("-" * 120)

candidate_display = ranked[
    ranked["audit_score"] > 0
][
    [
        "audit_score",
        "rows",
        "path",
        "relevant_fields",
    ]
].head(20)

if len(candidate_display):
    print(candidate_display.to_string(index=False))
else:
    print("No candidate CSV with relevant fields found.")

# ------------------------------------------------------------
# Specific check:
# Do any frozen CSVs contain ALL FIVE parsed KPI slots?
# ------------------------------------------------------------

parsed_slots = {
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
}

full_slot_files = []

for path in csv_files:

    try:
        header = set(
            pd.read_csv(
                path,
                nrows=0
            ).columns
        )
    except Exception:
        continue

    if parsed_slots.issubset(header):
        full_slot_files.append(path)

print("\nFILES CONTAINING ALL FIVE PARSED KPI SLOTS")
print("-" * 120)

if full_slot_files:
    for p in full_slot_files:
        print(p.relative_to(BASE_ROOT))
else:
    print("NONE")

# ------------------------------------------------------------
# Verify frozen source again
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

assert sha256_file(SLICE_SOURCE) == EXPECTED_SOURCE_HASH

print("\nInventory saved:")
print(inventory_path)

print("\n✓ Frozen artifact inventory complete")
print("✓ No experiment rerun performed")
print("✓ No result file modified")
print("✓ Frozen E1-E5 source unchanged")

R2-18 — DISCOVER FROZEN AUDIT INPUT ARTIFACTS

CSV files found  : 24
JSON files found : 16

CSV ARTIFACTS WITH POTENTIALLY RELEVANT COLUMNS
------------------------------------------------------------------------------------------------------------------------

FILE:
01_E1_E4_5SEED_FINAL/e1_models.csv
Shape : (7, 7)
Relevant columns:
['valid']

FILE:
01_E1_E4_5SEED_FINAL/e3_variants.csv
Shape : (2, 11)
Relevant columns:
['escalated80']

FILE:
01_E1_E4_5SEED_FINAL/e4_factorial.csv
Shape : (4, 18)
Relevant columns:
['escalated80']

FILE:
01_E1_E4_5SEED_FINAL/seeds/e1_models_seed42.csv
Shape : (7, 7)
Relevant columns:
['valid']

FILE:
01_E1_E4_5SEED_FINAL/seeds/e1_models_seed43.csv
Shape : (7, 7)
Relevant columns:
['valid']

FILE:
01_E1_E4_5SEED_FINAL/seeds/e1_models_seed44.csv
Shape : (7, 7)
Relevant columns:
['valid']

FILE:
01_E1_E4_5SEED_FINAL/seeds/e1_models_seed45.csv
Shape : (7, 7)
Relevant columns:
['valid']

FILE:
01_E1_E4_5SEED_FINAL/seeds/e1_models_seed46.csv
Shape : (7, 7)
Rel

In [91]:
# ============================================================
# R2-19 — RECONSTRUCT FROZEN PARSED KPI SLOTS
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import json

print("=" * 120)
print("R2-19 — RECONSTRUCT PARSED KPI SLOTS FROM FROZEN INTENT TEXT")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_RAW.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "corpus" in globals(), (
    "STOP: frozen corpus is not loaded. "
    "R2-17 must be completed first."
)

assert "CORPUS_PATH" in globals()
assert "CORPUS_SHA256" in globals()

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert CORPUS_SHA256 == EXPECTED_CORPUS_HASH, (
    "STOP: unexpected frozen corpus SHA-256"
)

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

assert sha256_file(SLICE_SOURCE) == EXPECTED_SOURCE_HASH


# ------------------------------------------------------------
# Reconstruct parsed slots using EXACT frozen parser
# ------------------------------------------------------------

parsed_rows = []

for _, source_row in corpus.iterrows():

    intent = frozen.parse_intent(
        source_row["text"]
    )

    parsed_rows.append({
        "id": source_row["id"],
        "text": source_row["text"],
        "true_slice": source_row["slice"],
        "uc_kind": source_row["uc_kind"],
        "template": source_row["template"],
        "ambiguous": source_row["ambiguous"],

        "latency_ms": intent.latency_ms,
        "throughput_mbps": intent.throughput_mbps,
        "reliability_pct": intent.reliability_pct,
        "density_per_km2": intent.density_per_km2,
        "mobility_kmh": intent.mobility_kmh,

        # Retain generated truth only for parser-audit comparison.
        # These are NOT substituted for parsed slots.
        "lat_true": source_row["lat_true"],
        "thr_true": source_row["thr_true"],
        "rel_true": source_row["rel_true"],
        "den_true": source_row["den_true"],
        "mob_true": source_row["mob_true"],
    })

parsed = pd.DataFrame(parsed_rows)

assert len(parsed) == len(corpus)
assert parsed["id"].tolist() == corpus["id"].tolist()


# ------------------------------------------------------------
# Parsed-slot availability
# ------------------------------------------------------------

slot_columns = [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
]

print("\nPARSED KPI SLOT AVAILABILITY")
print("-" * 120)

availability_rows = []

for col in slot_columns:

    present = int(
        parsed[col].notna().sum()
    )

    missing = int(
        parsed[col].isna().sum()
    )

    availability_rows.append({
        "kpi": col,
        "present": present,
        "missing": missing,
        "present_pct": 100.0 * present / len(parsed),
    })

availability = pd.DataFrame(
    availability_rows
)

print(
    availability.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


# ------------------------------------------------------------
# Parser-vs-generated-truth consistency
#
# Only compare rows where BOTH are present.
# Formatting in natural-language text can cause small numeric
# differences, so report exact/close agreement separately.
# ------------------------------------------------------------

pairs = [
    ("latency_ms", "lat_true"),
    ("throughput_mbps", "thr_true"),
    ("reliability_pct", "rel_true"),
    ("density_per_km2", "den_true"),
    ("mobility_kmh", "mob_true"),
]

comparison_rows = []

for parsed_col, truth_col in pairs:

    mask = (
        parsed[parsed_col].notna()
        &
        parsed[truth_col].notna()
    )

    a = parsed.loc[
        mask,
        parsed_col
    ].astype(float)

    b = parsed.loc[
        mask,
        truth_col
    ].astype(float)

    n = int(mask.sum())

    if n:

        exact = int(
            np.isclose(
                a,
                b,
                rtol=0.0,
                atol=1e-12
            ).sum()
        )

        close = int(
            np.isclose(
                a,
                b,
                rtol=0.01,
                atol=0.01
            ).sum()
        )

        mean_abs_error = float(
            np.mean(
                np.abs(a - b)
            )
        )

    else:

        exact = 0
        close = 0
        mean_abs_error = np.nan

    comparison_rows.append({
        "parsed_slot": parsed_col,
        "truth_column": truth_col,
        "comparable_rows": n,
        "exact_matches": exact,
        "close_1pct_matches": close,
        "close_1pct_pct": (
            100.0 * close / n
            if n
            else np.nan
        ),
        "mean_abs_error": mean_abs_error,
    })

parser_comparison = pd.DataFrame(
    comparison_rows
)

print("\nPARSER VS GENERATED-TRUTH CHECK")
print("-" * 120)

print(
    parser_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# Run BOTH feasibility implementations on reconstructed slots
# ------------------------------------------------------------

legacy_sets = []
strict_sets = []

for _, row in parsed.iterrows():

    legacy = frozen.feasible_set(row)
    strict = strict_feasible_set(row)

    legacy_sets.append(
        "|".join(legacy)
    )

    strict_sets.append(
        "|".join(strict)
    )

parsed["legacy_feasible_set"] = legacy_sets
parsed["strict_feasible_set"] = strict_sets

parsed["strict_empty"] = (
    parsed["strict_feasible_set"] == ""
)

parsed["legacy_strict_disagreement"] = (
    parsed["legacy_feasible_set"]
    !=
    parsed["strict_feasible_set"]
)


# ------------------------------------------------------------
# Important: DO NOT yet call these "false-feasible decisions".
#
# At this point we are auditing candidate sets only.
# Prediction-level validity comes later.
# ------------------------------------------------------------

print("\nCANDIDATE-SET PREVIEW")
print("-" * 120)

print(
    parsed[
        [
            "id",
            "true_slice",
            "legacy_feasible_set",
            "strict_feasible_set",
            "strict_empty",
            "legacy_strict_disagreement",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nPRELIMINARY CANDIDATE-SET COUNTS")
print("-" * 120)

print("Rows audited                 :", len(parsed))

print(
    "Legacy/strict disagreements :",
    int(
        parsed[
            "legacy_strict_disagreement"
        ].sum()
    )
)

print(
    "Strict empty feasible sets   :",
    int(
        parsed[
            "strict_empty"
        ].sum()
    )
)


# ------------------------------------------------------------
# Save evidence
# ------------------------------------------------------------

parsed_path = (
    RESULTS_RAW /
    "R2_reconstructed_parsed_slots.csv"
)

comparison_path = (
    AUDIT_DIR /
    "R2_parser_truth_consistency.csv"
)

availability_path = (
    AUDIT_DIR /
    "R2_parsed_slot_availability.csv"
)

parsed.to_csv(
    parsed_path,
    index=False
)

parser_comparison.to_csv(
    comparison_path,
    index=False
)

availability.to_csv(
    availability_path,
    index=False
)


# ------------------------------------------------------------
# Provenance record
# ------------------------------------------------------------

provenance = {
    "experiment":
        "R2 strict feasibility audit",

    "input_corpus":
        str(CORPUS_PATH),

    "input_corpus_sha256":
        CORPUS_SHA256,

    "parser_source":
        str(SLICE_SOURCE),

    "parser_source_sha256":
        sha256_file(SLICE_SOURCE),

    "parser_function":
        "parse_intent",

    "rows":
        int(len(parsed)),

    "method":
        (
            "Parsed KPI slots reconstructed deterministically "
            "from frozen intent text using the unchanged frozen "
            "parse_intent() implementation."
        ),

    "truth_columns_usage":
        (
            "Retained only for parser-consistency analysis; "
            "not substituted for parsed KPI slots."
        ),

    "experiment_rerun":
        False,
}

provenance_json = (
    AUDIT_DIR /
    "R2_parsed_slot_reconstruction_provenance.json"
)

with open(
    provenance_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        provenance,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Hash generated R2 evidence
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    parsed_path,
    comparison_path,
    availability_path,
    provenance_json,
]:

    print(p)
    print("SHA256:", file_sha256(p))


# ------------------------------------------------------------
# Final integrity check
# ------------------------------------------------------------

assert sha256_file(
    SLICE_SOURCE
) == EXPECTED_SOURCE_HASH

assert file_sha256(
    CORPUS_PATH
) == EXPECTED_CORPUS_HASH

print("\n✓ Parsed slots reconstructed using frozen parser")
print("✓ Generated truth not substituted for parser output")
print("✓ Legacy and strict candidate sets recorded")
print("✓ No model training performed")
print("✓ No E1-E5 experiment rerun performed")
print("✓ Frozen source and corpus remain unchanged")

R2-19 — RECONSTRUCT PARSED KPI SLOTS FROM FROZEN INTENT TEXT

PARSED KPI SLOT AVAILABILITY
------------------------------------------------------------------------------------------------------------------------
            kpi  present  missing  present_pct
     latency_ms      846      354        70.50
throughput_mbps      875      325        72.92
reliability_pct     1049      151        87.42
density_per_km2      674      526        56.17
   mobility_kmh      554      646        46.17

PARSER VS GENERATED-TRUTH CHECK
------------------------------------------------------------------------------------------------------------------------
    parsed_slot truth_column  comparable_rows  exact_matches  close_1pct_matches  close_1pct_pct  mean_abs_error
     latency_ms     lat_true              846              0                 697       82.387707        0.184793
throughput_mbps     thr_true              875              0                 787       89.942857        0.108947
reliability_p

In [92]:
# ============================================================
# R2-19 — RECONSTRUCT FROZEN PARSED KPI SLOTS
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import json

print("=" * 120)
print("R2-19 — RECONSTRUCT PARSED KPI SLOTS FROM FROZEN INTENT TEXT")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_RAW.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "corpus" in globals(), (
    "STOP: frozen corpus is not loaded. "
    "R2-17 must be completed first."
)

assert "CORPUS_PATH" in globals()
assert "CORPUS_SHA256" in globals()

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert CORPUS_SHA256 == EXPECTED_CORPUS_HASH, (
    "STOP: unexpected frozen corpus SHA-256"
)

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

assert sha256_file(SLICE_SOURCE) == EXPECTED_SOURCE_HASH


# ------------------------------------------------------------
# Reconstruct parsed slots using EXACT frozen parser
# ------------------------------------------------------------

parsed_rows = []

for _, source_row in corpus.iterrows():

    intent = frozen.parse_intent(
        source_row["text"]
    )

    parsed_rows.append({
        "id": source_row["id"],
        "text": source_row["text"],
        "true_slice": source_row["slice"],
        "uc_kind": source_row["uc_kind"],
        "template": source_row["template"],
        "ambiguous": source_row["ambiguous"],

        "latency_ms": intent.latency_ms,
        "throughput_mbps": intent.throughput_mbps,
        "reliability_pct": intent.reliability_pct,
        "density_per_km2": intent.density_per_km2,
        "mobility_kmh": intent.mobility_kmh,

        # Retain generated truth only for parser-audit comparison.
        # These are NOT substituted for parsed slots.
        "lat_true": source_row["lat_true"],
        "thr_true": source_row["thr_true"],
        "rel_true": source_row["rel_true"],
        "den_true": source_row["den_true"],
        "mob_true": source_row["mob_true"],
    })

parsed = pd.DataFrame(parsed_rows)

assert len(parsed) == len(corpus)
assert parsed["id"].tolist() == corpus["id"].tolist()


# ------------------------------------------------------------
# Parsed-slot availability
# ------------------------------------------------------------

slot_columns = [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
]

print("\nPARSED KPI SLOT AVAILABILITY")
print("-" * 120)

availability_rows = []

for col in slot_columns:

    present = int(
        parsed[col].notna().sum()
    )

    missing = int(
        parsed[col].isna().sum()
    )

    availability_rows.append({
        "kpi": col,
        "present": present,
        "missing": missing,
        "present_pct": 100.0 * present / len(parsed),
    })

availability = pd.DataFrame(
    availability_rows
)

print(
    availability.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


# ------------------------------------------------------------
# Parser-vs-generated-truth consistency
#
# Only compare rows where BOTH are present.
# Formatting in natural-language text can cause small numeric
# differences, so report exact/close agreement separately.
# ------------------------------------------------------------

pairs = [
    ("latency_ms", "lat_true"),
    ("throughput_mbps", "thr_true"),
    ("reliability_pct", "rel_true"),
    ("density_per_km2", "den_true"),
    ("mobility_kmh", "mob_true"),
]

comparison_rows = []

for parsed_col, truth_col in pairs:

    mask = (
        parsed[parsed_col].notna()
        &
        parsed[truth_col].notna()
    )

    a = parsed.loc[
        mask,
        parsed_col
    ].astype(float)

    b = parsed.loc[
        mask,
        truth_col
    ].astype(float)

    n = int(mask.sum())

    if n:

        exact = int(
            np.isclose(
                a,
                b,
                rtol=0.0,
                atol=1e-12
            ).sum()
        )

        close = int(
            np.isclose(
                a,
                b,
                rtol=0.01,
                atol=0.01
            ).sum()
        )

        mean_abs_error = float(
            np.mean(
                np.abs(a - b)
            )
        )

    else:

        exact = 0
        close = 0
        mean_abs_error = np.nan

    comparison_rows.append({
        "parsed_slot": parsed_col,
        "truth_column": truth_col,
        "comparable_rows": n,
        "exact_matches": exact,
        "close_1pct_matches": close,
        "close_1pct_pct": (
            100.0 * close / n
            if n
            else np.nan
        ),
        "mean_abs_error": mean_abs_error,
    })

parser_comparison = pd.DataFrame(
    comparison_rows
)

print("\nPARSER VS GENERATED-TRUTH CHECK")
print("-" * 120)

print(
    parser_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# Run BOTH feasibility implementations on reconstructed slots
# ------------------------------------------------------------

legacy_sets = []
strict_sets = []

for _, row in parsed.iterrows():

    legacy = frozen.feasible_set(row)
    strict = strict_feasible_set(row)

    legacy_sets.append(
        "|".join(legacy)
    )

    strict_sets.append(
        "|".join(strict)
    )

parsed["legacy_feasible_set"] = legacy_sets
parsed["strict_feasible_set"] = strict_sets

parsed["strict_empty"] = (
    parsed["strict_feasible_set"] == ""
)

parsed["legacy_strict_disagreement"] = (
    parsed["legacy_feasible_set"]
    !=
    parsed["strict_feasible_set"]
)


# ------------------------------------------------------------
# Important: DO NOT yet call these "false-feasible decisions".
#
# At this point we are auditing candidate sets only.
# Prediction-level validity comes later.
# ------------------------------------------------------------

print("\nCANDIDATE-SET PREVIEW")
print("-" * 120)

print(
    parsed[
        [
            "id",
            "true_slice",
            "legacy_feasible_set",
            "strict_feasible_set",
            "strict_empty",
            "legacy_strict_disagreement",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nPRELIMINARY CANDIDATE-SET COUNTS")
print("-" * 120)

print("Rows audited                 :", len(parsed))

print(
    "Legacy/strict disagreements :",
    int(
        parsed[
            "legacy_strict_disagreement"
        ].sum()
    )
)

print(
    "Strict empty feasible sets   :",
    int(
        parsed[
            "strict_empty"
        ].sum()
    )
)


# ------------------------------------------------------------
# Save evidence
# ------------------------------------------------------------

parsed_path = (
    RESULTS_RAW /
    "R2_reconstructed_parsed_slots.csv"
)

comparison_path = (
    AUDIT_DIR /
    "R2_parser_truth_consistency.csv"
)

availability_path = (
    AUDIT_DIR /
    "R2_parsed_slot_availability.csv"
)

parsed.to_csv(
    parsed_path,
    index=False
)

parser_comparison.to_csv(
    comparison_path,
    index=False
)

availability.to_csv(
    availability_path,
    index=False
)


# ------------------------------------------------------------
# Provenance record
# ------------------------------------------------------------

provenance = {
    "experiment":
        "R2 strict feasibility audit",

    "input_corpus":
        str(CORPUS_PATH),

    "input_corpus_sha256":
        CORPUS_SHA256,

    "parser_source":
        str(SLICE_SOURCE),

    "parser_source_sha256":
        sha256_file(SLICE_SOURCE),

    "parser_function":
        "parse_intent",

    "rows":
        int(len(parsed)),

    "method":
        (
            "Parsed KPI slots reconstructed deterministically "
            "from frozen intent text using the unchanged frozen "
            "parse_intent() implementation."
        ),

    "truth_columns_usage":
        (
            "Retained only for parser-consistency analysis; "
            "not substituted for parsed KPI slots."
        ),

    "experiment_rerun":
        False,
}

provenance_json = (
    AUDIT_DIR /
    "R2_parsed_slot_reconstruction_provenance.json"
)

with open(
    provenance_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        provenance,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Hash generated R2 evidence
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    parsed_path,
    comparison_path,
    availability_path,
    provenance_json,
]:

    print(p)
    print("SHA256:", file_sha256(p))


# ------------------------------------------------------------
# Final integrity check
# ------------------------------------------------------------

assert sha256_file(
    SLICE_SOURCE
) == EXPECTED_SOURCE_HASH

assert file_sha256(
    CORPUS_PATH
) == EXPECTED_CORPUS_HASH

print("\n✓ Parsed slots reconstructed using frozen parser")
print("✓ Generated truth not substituted for parser output")
print("✓ Legacy and strict candidate sets recorded")
print("✓ No model training performed")
print("✓ No E1-E5 experiment rerun performed")
print("✓ Frozen source and corpus remain unchanged")

R2-19 — RECONSTRUCT PARSED KPI SLOTS FROM FROZEN INTENT TEXT

PARSED KPI SLOT AVAILABILITY
------------------------------------------------------------------------------------------------------------------------
            kpi  present  missing  present_pct
     latency_ms      846      354        70.50
throughput_mbps      875      325        72.92
reliability_pct     1049      151        87.42
density_per_km2      674      526        56.17
   mobility_kmh      554      646        46.17

PARSER VS GENERATED-TRUTH CHECK
------------------------------------------------------------------------------------------------------------------------
    parsed_slot truth_column  comparable_rows  exact_matches  close_1pct_matches  close_1pct_pct  mean_abs_error
     latency_ms     lat_true              846              0                 697       82.387707        0.184793
throughput_mbps     thr_true              875              0                 787       89.942857        0.108947
reliability_p

In [93]:
# ============================================================
# R2-20 — KPI-LEVEL LEGACY VS STRICT DISAGREEMENT ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
import json

print("=" * 120)
print("R2-20 — KPI-LEVEL LEGACY VS STRICT FEASIBILITY ANALYSIS")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_AGG = ROOT / "04_RESULTS_AGGREGATE"

RESULTS_RAW.mkdir(parents=True, exist_ok=True)
RESULTS_AGG.mkdir(parents=True, exist_ok=True)

assert "parsed" in globals()
assert len(parsed) == 1200

# ------------------------------------------------------------
# Helper: determine strict violations for every slice
# ------------------------------------------------------------

detail_rows = []

for _, row in parsed.iterrows():

    legacy_set = frozen.feasible_set(row)
    strict_set = strict_feasible_set(row)

    legacy_set_s = set(legacy_set)
    strict_set_s = set(strict_set)

    # Candidates allowed by legacy but rejected by strict
    legacy_extra = [
        s for s in frozen.SLICES
        if s in legacy_set_s
        and s not in strict_set_s
    ]

    # Candidates rejected by legacy but allowed by strict
    legacy_missing = [
        s for s in frozen.SLICES
        if s in strict_set_s
        and s not in legacy_set_s
    ]

    # Collect strict violations per slice.
    violations_by_slice = {}

    for slice_name in frozen.SLICES:

        audit = strict_slice_audit(
            slice_name,
            row
        )

        violations_by_slice[slice_name] = (
            audit["violations"]
        )

    # Why were legacy-extra candidates invalid under strict?
    extra_violation_kpis = set()

    for slice_name in legacy_extra:
        extra_violation_kpis.update(
            violations_by_slice[slice_name]
        )

    mobility_caused_extra = (
        "mobility_kmh" in extra_violation_kpis
    )

    # Mobility-only means every legacy-extra candidate is
    # rejected strictly solely because of mobility.
    mobility_only = False

    if legacy_extra:

        rejected_reasons = [
            set(
                violations_by_slice[s]
            )
            for s in legacy_extra
        ]

        mobility_only = all(
            reasons == {"mobility_kmh"}
            for reasons in rejected_reasons
        )

    detail_rows.append({

        "id":
            row["id"],

        "true_slice":
            row["true_slice"],

        "uc_kind":
            row["uc_kind"],

        "template":
            row["template"],

        "ambiguous":
            row["ambiguous"],

        "latency_ms":
            row["latency_ms"],

        "throughput_mbps":
            row["throughput_mbps"],

        "reliability_pct":
            row["reliability_pct"],

        "density_per_km2":
            row["density_per_km2"],

        "mobility_kmh":
            row["mobility_kmh"],

        "legacy_feasible_set":
            "|".join(legacy_set),

        "strict_feasible_set":
            "|".join(strict_set),

        "legacy_strict_disagreement":
            legacy_set != strict_set,

        "strict_empty":
            len(strict_set) == 0,

        "legacy_extra_candidates":
            "|".join(legacy_extra),

        "legacy_missing_candidates":
            "|".join(legacy_missing),

        "legacy_extra_count":
            len(legacy_extra),

        "legacy_missing_count":
            len(legacy_missing),

        "extra_violation_kpis":
            "|".join(
                sorted(extra_violation_kpis)
            ),

        "mobility_contributed":
            mobility_caused_extra,

        "mobility_only_disagreement":
            mobility_only,

        "violations_eMBB":
            "|".join(
                violations_by_slice["eMBB"]
            ),

        "violations_URLLC":
            "|".join(
                violations_by_slice["URLLC"]
            ),

        "violations_mMTC":
            "|".join(
                violations_by_slice["mMTC"]
            ),
    })


detail = pd.DataFrame(detail_rows)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(detail) == 1200

assert int(
    detail[
        "legacy_strict_disagreement"
    ].sum()
) == 541

assert int(
    detail[
        "strict_empty"
    ].sum()
) == 31


# ------------------------------------------------------------
# KPI attribution
#
# Count a row once for a KPI if at least one candidate that
# legacy incorrectly retained violates that KPI under strict.
# Counts can overlap because one row may violate multiple KPIs.
# ------------------------------------------------------------

kpis = [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
]

kpi_rows = []

for kpi in kpis:

    mask = detail[
        "extra_violation_kpis"
    ].fillna("").apply(
        lambda x:
        kpi in x.split("|")
        if x else False
    )

    count = int(mask.sum())

    kpi_rows.append({
        "kpi": kpi,
        "rows_contributing_to_legacy_overacceptance":
            count,
        "pct_of_all_1200":
            100.0 * count / len(detail),
        "pct_of_541_disagreements":
            100.0 * count / 541,
    })

kpi_summary = pd.DataFrame(kpi_rows)


# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

n = len(detail)

n_disagree = int(
    detail[
        "legacy_strict_disagreement"
    ].sum()
)

n_empty = int(
    detail[
        "strict_empty"
    ].sum()
)

n_extra = int(
    (
        detail[
            "legacy_extra_count"
        ] > 0
    ).sum()
)

n_missing = int(
    (
        detail[
            "legacy_missing_count"
        ] > 0
    ).sum()
)

n_mobility = int(
    detail[
        "mobility_contributed"
    ].sum()
)

n_mobility_only = int(
    detail[
        "mobility_only_disagreement"
    ].sum()
)


summary = pd.DataFrame([
    {
        "metric":
            "rows_audited",
        "count":
            n,
        "pct":
            100.0,
    },
    {
        "metric":
            "legacy_strict_disagreement",
        "count":
            n_disagree,
        "pct":
            100.0 * n_disagree / n,
    },
    {
        "metric":
            "legacy_retained_strictly_invalid_candidate",
        "count":
            n_extra,
        "pct":
            100.0 * n_extra / n,
    },
    {
        "metric":
            "legacy_rejected_strictly_valid_candidate",
        "count":
            n_missing,
        "pct":
            100.0 * n_missing / n,
    },
    {
        "metric":
            "strict_empty_feasible_set",
        "count":
            n_empty,
        "pct":
            100.0 * n_empty / n,
    },
    {
        "metric":
            "mobility_contributed_to_overacceptance",
        "count":
            n_mobility,
        "pct":
            100.0 * n_mobility / n,
    },
    {
        "metric":
            "mobility_only_disagreement",
        "count":
            n_mobility_only,
        "pct":
            100.0 * n_mobility_only / n,
    },
])


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nOVERALL SUMMARY")
print("-" * 120)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\nKPI ATTRIBUTION")
print("-" * 120)

print(
    kpi_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ------------------------------------------------------------
# Strict-empty-set analysis
# ------------------------------------------------------------

empty_cases = detail[
    detail["strict_empty"]
].copy()

print("\nSTRICT EMPTY-SET CASES")
print("-" * 120)

print(
    empty_cases[
        [
            "id",
            "true_slice",
            "legacy_feasible_set",
            "extra_violation_kpis",
            "mobility_kmh",
        ]
    ]
    .head(31)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Mobility-only examples
# ------------------------------------------------------------

mobility_only_cases = detail[
    detail[
        "mobility_only_disagreement"
    ]
].copy()

print("\nMOBILITY-ONLY DISAGREEMENT EXAMPLES")
print("-" * 120)

if len(mobility_only_cases):

    print(
        mobility_only_cases[
            [
                "id",
                "true_slice",
                "mobility_kmh",
                "legacy_feasible_set",
                "strict_feasible_set",
                "legacy_extra_candidates",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

else:
    print("NONE")


# ------------------------------------------------------------
# Save evidence
# ------------------------------------------------------------

detail_path = (
    RESULTS_RAW /
    "R2_per_intent_feasibility_audit.csv"
)

summary_path = (
    RESULTS_AGG /
    "R2_feasibility_summary.csv"
)

kpi_path = (
    RESULTS_AGG /
    "R2_kpi_attribution.csv"
)

empty_path = (
    RESULTS_RAW /
    "R2_strict_empty_cases.csv"
)

mobility_path = (
    RESULTS_RAW /
    "R2_mobility_only_disagreements.csv"
)

detail.to_csv(
    detail_path,
    index=False
)

summary.to_csv(
    summary_path,
    index=False
)

kpi_summary.to_csv(
    kpi_path,
    index=False
)

empty_cases.to_csv(
    empty_path,
    index=False
)

mobility_only_cases.to_csv(
    mobility_path,
    index=False
)


# ------------------------------------------------------------
# Hash outputs
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    detail_path,
    summary_path,
    kpi_path,
    empty_path,
    mobility_path,
]:

    print(p)
    print("SHA256:", file_sha256(p))


# ------------------------------------------------------------
# Frozen evidence verification
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

assert (
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

print("\n✓ KPI-level disagreement analysis complete")
print("✓ Mobility contribution isolated")
print("✓ Strict empty-set cases preserved")
print("✓ No prediction-level false-feasible claim made yet")
print("✓ Frozen source and corpus unchanged")

R2-20 — KPI-LEVEL LEGACY VS STRICT FEASIBILITY ANALYSIS

OVERALL SUMMARY
------------------------------------------------------------------------------------------------------------------------
                                    metric  count     pct
                              rows_audited   1200 100.000
                legacy_strict_disagreement    541  45.083
legacy_retained_strictly_invalid_candidate    526  43.833
  legacy_rejected_strictly_valid_candidate     36   3.000
                 strict_empty_feasible_set     31   2.583
    mobility_contributed_to_overacceptance     61   5.083
                mobility_only_disagreement     12   1.000

KPI ATTRIBUTION
------------------------------------------------------------------------------------------------------------------------
            kpi  rows_contributing_to_legacy_overacceptance  pct_of_all_1200  pct_of_541_disagreements
     latency_ms                                         443           36.917                    81.88

In [94]:
# ============================================================
# R2-21A — STRICT ORACLE V2: CORRECT LATENCY DIRECTION
# ============================================================

import numpy as np
import hashlib
import inspect
from pathlib import Path

print("=" * 120)
print("R2-21A — STRICT FEASIBILITY ORACLE V2")
print("CORRECT LATENCY REQUIREMENT DIRECTION")
print("=" * 120)

AUDIT_DIR = ROOT / "02_R2_AUDIT"

# ------------------------------------------------------------
# Strict requirement semantics
#
# latency:
#   request means "latency must be <= X"
#   slice feasible when its BEST supported latency <= X
#   catalogue_min <= requested_latency
#
# throughput/reliability/density/mobility:
#   request is required capability
#   slice feasible when request <= catalogue_max
# ------------------------------------------------------------

KPI_RULES_V2 = {
    "latency_ms": {
        "catalogue_key": "lat",
        "comparison": "catalogue_min<=request",
    },
    "throughput_mbps": {
        "catalogue_key": "thr",
        "comparison": "request<=catalogue_max",
    },
    "reliability_pct": {
        "catalogue_key": "rel",
        "comparison": "request<=catalogue_max",
    },
    "density_per_km2": {
        "catalogue_key": "den",
        "comparison": "request<=catalogue_max",
    },
    "mobility_kmh": {
        "catalogue_key": "mob",
        "comparison": "request<=catalogue_max",
    },
}


def strict_slice_audit_v2(slice_name, row):

    c = frozen.CATALOGUE[slice_name]

    violations = []
    checks = {}

    for kpi, rule in KPI_RULES_V2.items():

        request_value = _finite_value(row, kpi)
        catalogue_key = rule["catalogue_key"]

        lower, upper = c[catalogue_key]

        if request_value is None:

            checks[kpi] = {
                "stated": False,
                "request": None,
                "catalogue_min": float(lower),
                "catalogue_max": float(upper),
                "pass": None,
            }

            continue

        # ----------------------------------------------------
        # Direction-aware feasibility
        # ----------------------------------------------------

        if kpi == "latency_ms":

            # Requested latency is an upper-bound SLA.
            # Slice must be capable of latency <= requested value.
            passed = float(lower) <= (
                float(request_value) + 1e-12
            )

        else:

            # Required capability must not exceed slice maximum.
            passed = float(request_value) <= (
                float(upper) + 1e-12
            )

        checks[kpi] = {
            "stated": True,
            "request": float(request_value),
            "catalogue_min": float(lower),
            "catalogue_max": float(upper),
            "pass": bool(passed),
        }

        if not passed:
            violations.append(kpi)

    return {
        "slice": slice_name,
        "feasible": len(violations) == 0,
        "violations": violations,
        "checks": checks,
    }


def strict_feasible_set_v2(row):

    feasible = []

    for slice_name in frozen.SLICES:

        audit = strict_slice_audit_v2(
            slice_name,
            row
        )

        if audit["feasible"]:
            feasible.append(slice_name)

    return feasible


def strict_contract_valid_v2(pred_label, row):

    return (
        pred_label
        in strict_feasible_set_v2(row)
    )


# ------------------------------------------------------------
# Critical directional sanity tests
# ------------------------------------------------------------

latency_1ms = {
    "latency_ms": 1.0,
    "throughput_mbps": np.nan,
    "reliability_pct": np.nan,
    "density_per_km2": np.nan,
    "mobility_kmh": np.nan,
}

latency_5ms = {
    "latency_ms": 5.0,
    "throughput_mbps": np.nan,
    "reliability_pct": np.nan,
    "density_per_km2": np.nan,
    "mobility_kmh": np.nan,
}

latency_20ms = {
    "latency_ms": 20.0,
    "throughput_mbps": np.nan,
    "reliability_pct": np.nan,
    "density_per_km2": np.nan,
    "mobility_kmh": np.nan,
}

print("\nLATENCY SANITY TESTS")
print("-" * 120)

print(
    "Request <= 1 ms  :",
    strict_feasible_set_v2(latency_1ms)
)

print(
    "Request <= 5 ms  :",
    strict_feasible_set_v2(latency_5ms)
)

print(
    "Request <= 20 ms :",
    strict_feasible_set_v2(latency_20ms)
)

# Expected:
# <=1 ms  -> URLLC
# <=5 ms  -> URLLC
# <=20 ms -> eMBB + URLLC

assert strict_feasible_set_v2(
    latency_1ms
) == ["URLLC"]

assert strict_feasible_set_v2(
    latency_5ms
) == ["URLLC"]

assert strict_feasible_set_v2(
    latency_20ms
) == ["eMBB", "URLLC"]


# ------------------------------------------------------------
# Mobility sanity tests remain unchanged
# ------------------------------------------------------------

mobility_100 = {
    "latency_ms": np.nan,
    "throughput_mbps": np.nan,
    "reliability_pct": np.nan,
    "density_per_km2": np.nan,
    "mobility_kmh": 100.0,
}

mobility_300 = {
    "latency_ms": np.nan,
    "throughput_mbps": np.nan,
    "reliability_pct": np.nan,
    "density_per_km2": np.nan,
    "mobility_kmh": 300.0,
}

mobility_600 = {
    "latency_ms": np.nan,
    "throughput_mbps": np.nan,
    "reliability_pct": np.nan,
    "density_per_km2": np.nan,
    "mobility_kmh": 600.0,
}

print("\nMOBILITY SANITY TESTS")
print("-" * 120)

print(
    "100 km/h :",
    strict_feasible_set_v2(mobility_100)
)

print(
    "300 km/h :",
    strict_feasible_set_v2(mobility_300)
)

print(
    "600 km/h :",
    strict_feasible_set_v2(mobility_600)
)

assert strict_feasible_set_v2(
    mobility_100
) == ["eMBB", "URLLC"]

assert strict_feasible_set_v2(
    mobility_300
) == ["URLLC"]

assert strict_feasible_set_v2(
    mobility_600
) == []


# ------------------------------------------------------------
# Impossible multi-KPI case must remain empty
# ------------------------------------------------------------

impossible_v2 = {
    "latency_ms": 0.1,
    "throughput_mbps": 5000.0,
    "reliability_pct": 99.99999,
    "density_per_km2": 2_000_000.0,
    "mobility_kmh": 1000.0,
}

legacy_impossible_v2 = frozen.feasible_set(
    impossible_v2
)

strict_impossible_v2 = strict_feasible_set_v2(
    impossible_v2
)

print("\nIMPOSSIBLE CASE")
print("-" * 120)

print(
    "Legacy feasible set :",
    legacy_impossible_v2
)

print(
    "Strict V2 set       :",
    strict_impossible_v2
)

assert strict_impossible_v2 == []


# ------------------------------------------------------------
# Save V2 oracle
# ------------------------------------------------------------

oracle_v2_source = (
    inspect.getsource(strict_slice_audit_v2)
    + "\n\n"
    + inspect.getsource(strict_feasible_set_v2)
    + "\n\n"
    + inspect.getsource(strict_contract_valid_v2)
)

oracle_v2_path = (
    AUDIT_DIR /
    "R2_strict_feasibility_oracle_v2.py"
)

oracle_v2_path.write_text(
    oracle_v2_source,
    encoding="utf-8"
)

oracle_v2_sha = hashlib.sha256(
    oracle_v2_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# Preserve V1 rather than overwrite it
# ------------------------------------------------------------

print("\nORACLE PROVENANCE")
print("-" * 120)

print(
    "V1 retained:",
    AUDIT_DIR /
    "R2_strict_feasibility_oracle.py"
)

print(
    "V2 saved   :",
    oracle_v2_path
)

print(
    "V2 SHA256  :",
    oracle_v2_sha
)


# ------------------------------------------------------------
# Verify original source still frozen
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

print("\nKPI RULES V2")
print("-" * 120)

for kpi, rule in KPI_RULES_V2.items():

    print(
        f"{kpi:20s} -> "
        f"{rule['catalogue_key']:4s} | "
        f"{rule['comparison']}"
    )

print("\n✓ Strict oracle V2 created")
print("✓ Latency direction corrected")
print("✓ Mobility rule retained")
print("✓ Empty feasible set retained")
print("✓ V1 evidence preserved for provenance")
print("✓ Frozen E1-E5 source unchanged")

R2-21A — STRICT FEASIBILITY ORACLE V2
CORRECT LATENCY REQUIREMENT DIRECTION

LATENCY SANITY TESTS
------------------------------------------------------------------------------------------------------------------------
Request <= 1 ms  : ['URLLC']
Request <= 5 ms  : ['URLLC']
Request <= 20 ms : ['eMBB', 'URLLC']

MOBILITY SANITY TESTS
------------------------------------------------------------------------------------------------------------------------
100 km/h : ['eMBB', 'URLLC']
300 km/h : ['URLLC']
600 km/h : []

IMPOSSIBLE CASE
------------------------------------------------------------------------------------------------------------------------
Legacy feasible set : ['eMBB', 'URLLC', 'mMTC']
Strict V2 set       : []

ORACLE PROVENANCE
------------------------------------------------------------------------------------------------------------------------
V1 retained: /content/xagentic_r2_work/02_R2_AUDIT/R2_strict_feasibility_oracle.py
V2 saved   : /content/xagentic_r2_work/02_R2

In [95]:
# ============================================================
# R2-21B — CONTROLLED VALIDATION OF STRICT ORACLE V2
# ============================================================

import pandas as pd
import numpy as np

print("=" * 120)
print("R2-21B — CONTROLLED VALIDATION OF STRICT FEASIBILITY ORACLE V2")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_RAW.mkdir(parents=True, exist_ok=True)

NAN = np.nan

# ------------------------------------------------------------
# Controlled cases
# ------------------------------------------------------------

cases = [

    # --------------------------------------------------------
    # Latency direction tests
    # --------------------------------------------------------

    {
        "case_id": "L01",
        "category": "latency_only",
        "row": {
            "latency_ms": 1.0,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["URLLC"],
    },

    {
        "case_id": "L02",
        "category": "latency_only",
        "row": {
            "latency_ms": 5.0,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["URLLC"],
    },

    {
        "case_id": "L03",
        "category": "latency_only",
        "row": {
            "latency_ms": 20.0,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["eMBB", "URLLC"],
    },

    {
        "case_id": "L04",
        "category": "latency_only",
        "row": {
            "latency_ms": 100.0,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["eMBB", "URLLC", "mMTC"],
    },

    # --------------------------------------------------------
    # Mobility tests
    # --------------------------------------------------------

    {
        "case_id": "M01",
        "category": "mobility_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": 20.0,
        },
        "expected": ["eMBB", "URLLC", "mMTC"],
    },

    {
        "case_id": "M02",
        "category": "mobility_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": 100.0,
        },
        "expected": ["eMBB", "URLLC"],
    },

    {
        "case_id": "M03",
        "category": "mobility_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": 300.0,
        },
        "expected": ["URLLC"],
    },

    {
        "case_id": "M04",
        "category": "mobility_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": 600.0,
        },
        "expected": [],
    },

    # --------------------------------------------------------
    # Other KPI capability tests
    # --------------------------------------------------------

    {
        "case_id": "T01",
        "category": "throughput_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": 500.0,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["eMBB"],
    },

    {
        "case_id": "R01",
        "category": "reliability_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": 99.95,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["URLLC"],
    },

    {
        "case_id": "D01",
        "category": "density_only",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": 500000.0,
            "mobility_kmh": NAN,
        },
        "expected": ["mMTC"],
    },

    # --------------------------------------------------------
    # Missing constraints
    # --------------------------------------------------------

    {
        "case_id": "N01",
        "category": "missing_all",
        "row": {
            "latency_ms": NAN,
            "throughput_mbps": NAN,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": ["eMBB", "URLLC", "mMTC"],
    },

    # --------------------------------------------------------
    # Cross-KPI feasible cases
    # --------------------------------------------------------

    {
        "case_id": "C01",
        "category": "cross_kpi_feasible",
        "row": {
            "latency_ms": 5.0,
            "throughput_mbps": 50.0,
            "reliability_pct": 99.995,
            "density_per_km2": NAN,
            "mobility_kmh": 300.0,
        },
        "expected": ["URLLC"],
    },

    {
        "case_id": "C02",
        "category": "cross_kpi_feasible",
        "row": {
            "latency_ms": 20.0,
            "throughput_mbps": 500.0,
            "reliability_pct": 99.5,
            "density_per_km2": NAN,
            "mobility_kmh": 100.0,
        },
        "expected": ["eMBB"],
    },

    # --------------------------------------------------------
    # Cross-KPI conflict:
    # latency requires URLLC but throughput requires eMBB.
    # No single slice satisfies both.
    # --------------------------------------------------------

    {
        "case_id": "X01",
        "category": "cross_kpi_conflict",
        "row": {
            "latency_ms": 5.0,
            "throughput_mbps": 500.0,
            "reliability_pct": NAN,
            "density_per_km2": NAN,
            "mobility_kmh": NAN,
        },
        "expected": [],
    },

    # --------------------------------------------------------
    # Impossible request
    # --------------------------------------------------------

    {
        "case_id": "X02",
        "category": "impossible",
        "row": {
            "latency_ms": 0.1,
            "throughput_mbps": 5000.0,
            "reliability_pct": 99.99999,
            "density_per_km2": 2000000.0,
            "mobility_kmh": 1000.0,
        },
        "expected": [],
    },
]


# ------------------------------------------------------------
# Execute controlled validation
# ------------------------------------------------------------

rows = []

for case in cases:

    row = case["row"]

    strict_set = strict_feasible_set_v2(row)
    legacy_set = frozen.feasible_set(row)

    expected = case["expected"]

    passed = strict_set == expected

    rows.append({
        "case_id":
            case["case_id"],

        "category":
            case["category"],

        "expected_strict_set":
            "|".join(expected),

        "strict_v2_set":
            "|".join(strict_set),

        "legacy_set":
            "|".join(legacy_set),

        "strict_test_passed":
            passed,

        "legacy_strict_differs":
            legacy_set != strict_set,

        "strict_empty":
            len(strict_set) == 0,

        "legacy_nonempty_when_strict_empty":
            (
                len(strict_set) == 0
                and
                len(legacy_set) > 0
            ),
    })


controlled_v2 = pd.DataFrame(rows)


# ------------------------------------------------------------
# Print complete test table
# ------------------------------------------------------------

print("\nCONTROLLED TEST RESULTS")
print("-" * 120)

print(
    controlled_v2.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n = len(controlled_v2)

passed = int(
    controlled_v2[
        "strict_test_passed"
    ].sum()
)

differs = int(
    controlled_v2[
        "legacy_strict_differs"
    ].sum()
)

unsafe_nonempty = int(
    controlled_v2[
        "legacy_nonempty_when_strict_empty"
    ].sum()
)

print("\nSUMMARY")
print("-" * 120)

print(
    "Controlled cases                  :",
    n
)

print(
    "Strict V2 oracle tests passed     :",
    f"{passed} / {n}"
)

print(
    "Legacy/strict disagreements       :",
    differs
)

print(
    "Legacy nonempty / strict empty    :",
    unsafe_nonempty
)


# ------------------------------------------------------------
# Mandatory assertions
# ------------------------------------------------------------

assert passed == n, (
    "STOP: Strict V2 oracle failed one or more "
    "controlled expectations."
)

assert (
    controlled_v2.loc[
        controlled_v2["case_id"] == "L01",
        "strict_v2_set"
    ].iloc[0]
    == "URLLC"
)

assert (
    controlled_v2.loc[
        controlled_v2["case_id"] == "X01",
        "strict_v2_set"
    ].iloc[0]
    == ""
)

assert (
    controlled_v2.loc[
        controlled_v2["case_id"] == "M04",
        "strict_v2_set"
    ].iloc[0]
    == ""
)


# ------------------------------------------------------------
# Save V2 controlled evidence
# ------------------------------------------------------------

controlled_path = (
    RESULTS_RAW /
    "R2_controlled_feasibility_cases_v2.csv"
)

controlled_v2.to_csv(
    controlled_path,
    index=False
)

print("\nSaved:")
print(controlled_path)

print(
    "SHA256:",
    file_sha256(controlled_path)
)


# ------------------------------------------------------------
# Frozen evidence verification
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

assert (
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

print("\n✓ Strict V2 controlled validation complete")
print("✓ Latency direction validated")
print("✓ Mobility coverage validated")
print("✓ Cross-KPI conflict validated")
print("✓ Empty feasible sets preserved")
print("✓ Frozen source and corpus unchanged")

R2-21B — CONTROLLED VALIDATION OF STRICT FEASIBILITY ORACLE V2

CONTROLLED TEST RESULTS
------------------------------------------------------------------------------------------------------------------------
case_id           category expected_strict_set   strict_v2_set      legacy_set  strict_test_passed  legacy_strict_differs  strict_empty  legacy_nonempty_when_strict_empty
    L01       latency_only               URLLC           URLLC           URLLC                True                  False         False                              False
    L02       latency_only               URLLC           URLLC      eMBB|URLLC                True                   True         False                              False
    L03       latency_only          eMBB|URLLC      eMBB|URLLC      eMBB|URLLC                True                  False         False                              False
    L04       latency_only     eMBB|URLLC|mMTC eMBB|URLLC|mMTC eMBB|URLLC|mMTC                True         

In [96]:
# ============================================================
# R2-22 — FULL 1,200-INTENT STRICT V2 FEASIBILITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 120)
print("R2-22 — FULL 1,200-INTENT STRICT V2 FEASIBILITY AUDIT")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_AGG = ROOT / "04_RESULTS_AGGREGATE"

RESULTS_RAW.mkdir(parents=True, exist_ok=True)
RESULTS_AGG.mkdir(parents=True, exist_ok=True)

assert "parsed" in globals()
assert len(parsed) == 1200

# ------------------------------------------------------------
# Audit every frozen reconstructed intent with V2
# ------------------------------------------------------------

audit_rows = []

for _, row in parsed.iterrows():

    legacy = frozen.feasible_set(row)
    strict = strict_feasible_set_v2(row)

    legacy_set = set(legacy)
    strict_set = set(strict)

    # Candidates accepted by legacy but rejected by strict V2.
    legacy_extra = [
        s for s in frozen.SLICES
        if s in legacy_set
        and s not in strict_set
    ]

    # Candidates rejected by legacy but accepted by strict V2.
    legacy_missing = [
        s for s in frozen.SLICES
        if s in strict_set
        and s not in legacy_set
    ]

    violations_by_slice = {}

    for slice_name in frozen.SLICES:

        result = strict_slice_audit_v2(
            slice_name,
            row
        )

        violations_by_slice[slice_name] = (
            result["violations"]
        )

    # --------------------------------------------------------
    # Determine why legacy retained invalid candidates
    # --------------------------------------------------------

    extra_violation_kpis = set()

    for slice_name in legacy_extra:

        extra_violation_kpis.update(
            violations_by_slice[slice_name]
        )

    mobility_contributed = (
        "mobility_kmh"
        in extra_violation_kpis
    )

    latency_contributed = (
        "latency_ms"
        in extra_violation_kpis
    )

    # Mobility-only:
    # every extra candidate is rejected solely due to mobility.
    mobility_only = False

    if legacy_extra:

        reasons = [
            set(
                violations_by_slice[s]
            )
            for s in legacy_extra
        ]

        mobility_only = all(
            r == {"mobility_kmh"}
            for r in reasons
        )

    # Latency-only analogous diagnostic.
    latency_only = False

    if legacy_extra:

        reasons = [
            set(
                violations_by_slice[s]
            )
            for s in legacy_extra
        ]

        latency_only = all(
            r == {"latency_ms"}
            for r in reasons
        )

    audit_rows.append({

        "id":
            row["id"],

        "true_slice":
            row["true_slice"],

        "uc_kind":
            row["uc_kind"],

        "template":
            row["template"],

        "ambiguous":
            row["ambiguous"],

        "latency_ms":
            row["latency_ms"],

        "throughput_mbps":
            row["throughput_mbps"],

        "reliability_pct":
            row["reliability_pct"],

        "density_per_km2":
            row["density_per_km2"],

        "mobility_kmh":
            row["mobility_kmh"],

        "legacy_feasible_set":
            "|".join(legacy),

        "strict_v2_feasible_set":
            "|".join(strict),

        "legacy_strict_v2_disagreement":
            legacy != strict,

        "strict_v2_empty":
            len(strict) == 0,

        "legacy_nonempty_strict_empty":
            (
                len(legacy) > 0
                and
                len(strict) == 0
            ),

        "legacy_extra_candidates":
            "|".join(legacy_extra),

        "legacy_missing_candidates":
            "|".join(legacy_missing),

        "legacy_extra_count":
            len(legacy_extra),

        "legacy_missing_count":
            len(legacy_missing),

        "extra_violation_kpis":
            "|".join(
                sorted(extra_violation_kpis)
            ),

        "mobility_contributed":
            mobility_contributed,

        "mobility_only_disagreement":
            mobility_only,

        "latency_contributed":
            latency_contributed,

        "latency_only_disagreement":
            latency_only,

        "violations_eMBB":
            "|".join(
                violations_by_slice["eMBB"]
            ),

        "violations_URLLC":
            "|".join(
                violations_by_slice["URLLC"]
            ),

        "violations_mMTC":
            "|".join(
                violations_by_slice["mMTC"]
            ),
    })


audit_v2 = pd.DataFrame(audit_rows)

assert len(audit_v2) == 1200


# ------------------------------------------------------------
# KPI attribution
# ------------------------------------------------------------

KPIS = [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
]

kpi_rows = []

for kpi in KPIS:

    mask = (
        audit_v2[
            "extra_violation_kpis"
        ]
        .fillna("")
        .apply(
            lambda x:
                kpi in x.split("|")
                if x else False
        )
    )

    count = int(mask.sum())

    kpi_rows.append({
        "kpi":
            kpi,

        "rows_with_legacy_overacceptance_due_to_kpi":
            count,

        "pct_of_all_rows":
            100.0 * count / len(audit_v2),
    })


kpi_v2 = pd.DataFrame(kpi_rows)


# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

N = len(audit_v2)

n_disagree = int(
    audit_v2[
        "legacy_strict_v2_disagreement"
    ].sum()
)

n_extra = int(
    (
        audit_v2[
            "legacy_extra_count"
        ] > 0
    ).sum()
)

n_missing = int(
    (
        audit_v2[
            "legacy_missing_count"
        ] > 0
    ).sum()
)

n_empty = int(
    audit_v2[
        "strict_v2_empty"
    ].sum()
)

n_unsafe_empty = int(
    audit_v2[
        "legacy_nonempty_strict_empty"
    ].sum()
)

n_mobility = int(
    audit_v2[
        "mobility_contributed"
    ].sum()
)

n_mobility_only = int(
    audit_v2[
        "mobility_only_disagreement"
    ].sum()
)

n_latency = int(
    audit_v2[
        "latency_contributed"
    ].sum()
)

n_latency_only = int(
    audit_v2[
        "latency_only_disagreement"
    ].sum()
)


summary_v2 = pd.DataFrame([

    {
        "metric": "rows_audited",
        "count": N,
        "pct": 100.0,
    },

    {
        "metric": "legacy_strict_v2_disagreement",
        "count": n_disagree,
        "pct": 100.0 * n_disagree / N,
    },

    {
        "metric":
            "legacy_retained_strictly_invalid_candidate",
        "count": n_extra,
        "pct": 100.0 * n_extra / N,
    },

    {
        "metric":
            "legacy_rejected_strictly_valid_candidate",
        "count": n_missing,
        "pct": 100.0 * n_missing / N,
    },

    {
        "metric": "strict_v2_empty_feasible_set",
        "count": n_empty,
        "pct": 100.0 * n_empty / N,
    },

    {
        "metric":
            "legacy_nonempty_when_strict_v2_empty",
        "count": n_unsafe_empty,
        "pct": 100.0 * n_unsafe_empty / N,
    },

    {
        "metric":
            "mobility_contributed_to_overacceptance",
        "count": n_mobility,
        "pct": 100.0 * n_mobility / N,
    },

    {
        "metric":
            "mobility_only_disagreement",
        "count": n_mobility_only,
        "pct": 100.0 * n_mobility_only / N,
    },

    {
        "metric":
            "latency_contributed_to_overacceptance",
        "count": n_latency,
        "pct": 100.0 * n_latency / N,
    },

    {
        "metric":
            "latency_only_disagreement",
        "count": n_latency_only,
        "pct": 100.0 * n_latency_only / N,
    },
])


# ------------------------------------------------------------
# Print main results
# ------------------------------------------------------------

print("\nSTRICT V2 OVERALL SUMMARY")
print("-" * 120)

print(
    summary_v2.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nSTRICT V2 KPI ATTRIBUTION")
print("-" * 120)

print(
    kpi_v2.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ------------------------------------------------------------
# Empty-set evidence
# ------------------------------------------------------------

empty_v2 = audit_v2[
    audit_v2["strict_v2_empty"]
].copy()

print("\nSTRICT V2 EMPTY-SET CASES")
print("-" * 120)

print(
    empty_v2[
        [
            "id",
            "true_slice",
            "legacy_feasible_set",
            "extra_violation_kpis",
            "latency_ms",
            "throughput_mbps",
            "density_per_km2",
            "mobility_kmh",
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Mobility-only evidence
# ------------------------------------------------------------

mobility_only_v2 = audit_v2[
    audit_v2[
        "mobility_only_disagreement"
    ]
].copy()

print("\nMOBILITY-ONLY DISAGREEMENTS")
print("-" * 120)

if len(mobility_only_v2):

    print(
        mobility_only_v2[
            [
                "id",
                "true_slice",
                "mobility_kmh",
                "legacy_feasible_set",
                "strict_v2_feasible_set",
                "legacy_extra_candidates",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

else:
    print("NONE")


# ------------------------------------------------------------
# Save V2 evidence separately
# Never overwrite provisional V1 files.
# ------------------------------------------------------------

audit_path = (
    RESULTS_RAW /
    "R2_per_intent_feasibility_audit_v2.csv"
)

summary_path = (
    RESULTS_AGG /
    "R2_feasibility_summary_v2.csv"
)

kpi_path = (
    RESULTS_AGG /
    "R2_kpi_attribution_v2.csv"
)

empty_path = (
    RESULTS_RAW /
    "R2_strict_empty_cases_v2.csv"
)

mobility_path = (
    RESULTS_RAW /
    "R2_mobility_only_disagreements_v2.csv"
)


audit_v2.to_csv(
    audit_path,
    index=False
)

summary_v2.to_csv(
    summary_path,
    index=False
)

kpi_v2.to_csv(
    kpi_path,
    index=False
)

empty_v2.to_csv(
    empty_path,
    index=False
)

mobility_only_v2.to_csv(
    mobility_path,
    index=False
)


# ------------------------------------------------------------
# Hash outputs
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    audit_path,
    summary_path,
    kpi_path,
    empty_path,
    mobility_path,
]:

    print(p)
    print(
        "SHA256:",
        file_sha256(p)
    )


# ------------------------------------------------------------
# Frozen evidence verification
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

assert (
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

print("\n✓ Full 1,200-intent V2 audit complete")
print("✓ Direction-aware latency semantics applied")
print("✓ All five represented KPI dimensions evaluated")
print("✓ Mobility omission quantified")
print("✓ Empty strict feasible sets preserved")
print("✓ Provisional V1 audit files retained")
print("✓ Frozen source and corpus unchanged")

R2-22 — FULL 1,200-INTENT STRICT V2 FEASIBILITY AUDIT

STRICT V2 OVERALL SUMMARY
------------------------------------------------------------------------------------------------------------------------
                                    metric  count     pct
                              rows_audited   1200 100.000
             legacy_strict_v2_disagreement    229  19.083
legacy_retained_strictly_invalid_candidate    229  19.083
  legacy_rejected_strictly_valid_candidate      0   0.000
              strict_v2_empty_feasible_set     14   1.167
      legacy_nonempty_when_strict_v2_empty     14   1.167
    mobility_contributed_to_overacceptance     61   5.083
                mobility_only_disagreement     37   3.083
     latency_contributed_to_overacceptance     63   5.250
                 latency_only_disagreement     48   4.000

STRICT V2 KPI ATTRIBUTION
------------------------------------------------------------------------------------------------------------------------
            

In [97]:
# ============================================================
# R2-22 — FULL 1,200-INTENT STRICT V2 FEASIBILITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 120)
print("R2-22 — FULL 1,200-INTENT STRICT V2 FEASIBILITY AUDIT")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_AGG = ROOT / "04_RESULTS_AGGREGATE"

RESULTS_RAW.mkdir(parents=True, exist_ok=True)
RESULTS_AGG.mkdir(parents=True, exist_ok=True)

assert "parsed" in globals()
assert len(parsed) == 1200

# ------------------------------------------------------------
# Audit every frozen reconstructed intent with V2
# ------------------------------------------------------------

audit_rows = []

for _, row in parsed.iterrows():

    legacy = frozen.feasible_set(row)
    strict = strict_feasible_set_v2(row)

    legacy_set = set(legacy)
    strict_set = set(strict)

    # Candidates accepted by legacy but rejected by strict V2.
    legacy_extra = [
        s for s in frozen.SLICES
        if s in legacy_set
        and s not in strict_set
    ]

    # Candidates rejected by legacy but accepted by strict V2.
    legacy_missing = [
        s for s in frozen.SLICES
        if s in strict_set
        and s not in legacy_set
    ]

    violations_by_slice = {}

    for slice_name in frozen.SLICES:

        result = strict_slice_audit_v2(
            slice_name,
            row
        )

        violations_by_slice[slice_name] = (
            result["violations"]
        )

    # --------------------------------------------------------
    # Determine why legacy retained invalid candidates
    # --------------------------------------------------------

    extra_violation_kpis = set()

    for slice_name in legacy_extra:

        extra_violation_kpis.update(
            violations_by_slice[slice_name]
        )

    mobility_contributed = (
        "mobility_kmh"
        in extra_violation_kpis
    )

    latency_contributed = (
        "latency_ms"
        in extra_violation_kpis
    )

    # Mobility-only:
    # every extra candidate is rejected solely due to mobility.
    mobility_only = False

    if legacy_extra:

        reasons = [
            set(
                violations_by_slice[s]
            )
            for s in legacy_extra
        ]

        mobility_only = all(
            r == {"mobility_kmh"}
            for r in reasons
        )

    # Latency-only analogous diagnostic.
    latency_only = False

    if legacy_extra:

        reasons = [
            set(
                violations_by_slice[s]
            )
            for s in legacy_extra
        ]

        latency_only = all(
            r == {"latency_ms"}
            for r in reasons
        )

    audit_rows.append({

        "id":
            row["id"],

        "true_slice":
            row["true_slice"],

        "uc_kind":
            row["uc_kind"],

        "template":
            row["template"],

        "ambiguous":
            row["ambiguous"],

        "latency_ms":
            row["latency_ms"],

        "throughput_mbps":
            row["throughput_mbps"],

        "reliability_pct":
            row["reliability_pct"],

        "density_per_km2":
            row["density_per_km2"],

        "mobility_kmh":
            row["mobility_kmh"],

        "legacy_feasible_set":
            "|".join(legacy),

        "strict_v2_feasible_set":
            "|".join(strict),

        "legacy_strict_v2_disagreement":
            legacy != strict,

        "strict_v2_empty":
            len(strict) == 0,

        "legacy_nonempty_strict_empty":
            (
                len(legacy) > 0
                and
                len(strict) == 0
            ),

        "legacy_extra_candidates":
            "|".join(legacy_extra),

        "legacy_missing_candidates":
            "|".join(legacy_missing),

        "legacy_extra_count":
            len(legacy_extra),

        "legacy_missing_count":
            len(legacy_missing),

        "extra_violation_kpis":
            "|".join(
                sorted(extra_violation_kpis)
            ),

        "mobility_contributed":
            mobility_contributed,

        "mobility_only_disagreement":
            mobility_only,

        "latency_contributed":
            latency_contributed,

        "latency_only_disagreement":
            latency_only,

        "violations_eMBB":
            "|".join(
                violations_by_slice["eMBB"]
            ),

        "violations_URLLC":
            "|".join(
                violations_by_slice["URLLC"]
            ),

        "violations_mMTC":
            "|".join(
                violations_by_slice["mMTC"]
            ),
    })


audit_v2 = pd.DataFrame(audit_rows)

assert len(audit_v2) == 1200


# ------------------------------------------------------------
# KPI attribution
# ------------------------------------------------------------

KPIS = [
    "latency_ms",
    "throughput_mbps",
    "reliability_pct",
    "density_per_km2",
    "mobility_kmh",
]

kpi_rows = []

for kpi in KPIS:

    mask = (
        audit_v2[
            "extra_violation_kpis"
        ]
        .fillna("")
        .apply(
            lambda x:
                kpi in x.split("|")
                if x else False
        )
    )

    count = int(mask.sum())

    kpi_rows.append({
        "kpi":
            kpi,

        "rows_with_legacy_overacceptance_due_to_kpi":
            count,

        "pct_of_all_rows":
            100.0 * count / len(audit_v2),
    })


kpi_v2 = pd.DataFrame(kpi_rows)


# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

N = len(audit_v2)

n_disagree = int(
    audit_v2[
        "legacy_strict_v2_disagreement"
    ].sum()
)

n_extra = int(
    (
        audit_v2[
            "legacy_extra_count"
        ] > 0
    ).sum()
)

n_missing = int(
    (
        audit_v2[
            "legacy_missing_count"
        ] > 0
    ).sum()
)

n_empty = int(
    audit_v2[
        "strict_v2_empty"
    ].sum()
)

n_unsafe_empty = int(
    audit_v2[
        "legacy_nonempty_strict_empty"
    ].sum()
)

n_mobility = int(
    audit_v2[
        "mobility_contributed"
    ].sum()
)

n_mobility_only = int(
    audit_v2[
        "mobility_only_disagreement"
    ].sum()
)

n_latency = int(
    audit_v2[
        "latency_contributed"
    ].sum()
)

n_latency_only = int(
    audit_v2[
        "latency_only_disagreement"
    ].sum()
)


summary_v2 = pd.DataFrame([

    {
        "metric": "rows_audited",
        "count": N,
        "pct": 100.0,
    },

    {
        "metric": "legacy_strict_v2_disagreement",
        "count": n_disagree,
        "pct": 100.0 * n_disagree / N,
    },

    {
        "metric":
            "legacy_retained_strictly_invalid_candidate",
        "count": n_extra,
        "pct": 100.0 * n_extra / N,
    },

    {
        "metric":
            "legacy_rejected_strictly_valid_candidate",
        "count": n_missing,
        "pct": 100.0 * n_missing / N,
    },

    {
        "metric": "strict_v2_empty_feasible_set",
        "count": n_empty,
        "pct": 100.0 * n_empty / N,
    },

    {
        "metric":
            "legacy_nonempty_when_strict_v2_empty",
        "count": n_unsafe_empty,
        "pct": 100.0 * n_unsafe_empty / N,
    },

    {
        "metric":
            "mobility_contributed_to_overacceptance",
        "count": n_mobility,
        "pct": 100.0 * n_mobility / N,
    },

    {
        "metric":
            "mobility_only_disagreement",
        "count": n_mobility_only,
        "pct": 100.0 * n_mobility_only / N,
    },

    {
        "metric":
            "latency_contributed_to_overacceptance",
        "count": n_latency,
        "pct": 100.0 * n_latency / N,
    },

    {
        "metric":
            "latency_only_disagreement",
        "count": n_latency_only,
        "pct": 100.0 * n_latency_only / N,
    },
])


# ------------------------------------------------------------
# Print main results
# ------------------------------------------------------------

print("\nSTRICT V2 OVERALL SUMMARY")
print("-" * 120)

print(
    summary_v2.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nSTRICT V2 KPI ATTRIBUTION")
print("-" * 120)

print(
    kpi_v2.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ------------------------------------------------------------
# Empty-set evidence
# ------------------------------------------------------------

empty_v2 = audit_v2[
    audit_v2["strict_v2_empty"]
].copy()

print("\nSTRICT V2 EMPTY-SET CASES")
print("-" * 120)

print(
    empty_v2[
        [
            "id",
            "true_slice",
            "legacy_feasible_set",
            "extra_violation_kpis",
            "latency_ms",
            "throughput_mbps",
            "density_per_km2",
            "mobility_kmh",
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Mobility-only evidence
# ------------------------------------------------------------

mobility_only_v2 = audit_v2[
    audit_v2[
        "mobility_only_disagreement"
    ]
].copy()

print("\nMOBILITY-ONLY DISAGREEMENTS")
print("-" * 120)

if len(mobility_only_v2):

    print(
        mobility_only_v2[
            [
                "id",
                "true_slice",
                "mobility_kmh",
                "legacy_feasible_set",
                "strict_v2_feasible_set",
                "legacy_extra_candidates",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

else:
    print("NONE")


# ------------------------------------------------------------
# Save V2 evidence separately
# Never overwrite provisional V1 files.
# ------------------------------------------------------------

audit_path = (
    RESULTS_RAW /
    "R2_per_intent_feasibility_audit_v2.csv"
)

summary_path = (
    RESULTS_AGG /
    "R2_feasibility_summary_v2.csv"
)

kpi_path = (
    RESULTS_AGG /
    "R2_kpi_attribution_v2.csv"
)

empty_path = (
    RESULTS_RAW /
    "R2_strict_empty_cases_v2.csv"
)

mobility_path = (
    RESULTS_RAW /
    "R2_mobility_only_disagreements_v2.csv"
)


audit_v2.to_csv(
    audit_path,
    index=False
)

summary_v2.to_csv(
    summary_path,
    index=False
)

kpi_v2.to_csv(
    kpi_path,
    index=False
)

empty_v2.to_csv(
    empty_path,
    index=False
)

mobility_only_v2.to_csv(
    mobility_path,
    index=False
)


# ------------------------------------------------------------
# Hash outputs
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    audit_path,
    summary_path,
    kpi_path,
    empty_path,
    mobility_path,
]:

    print(p)
    print(
        "SHA256:",
        file_sha256(p)
    )


# ------------------------------------------------------------
# Frozen evidence verification
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

assert (
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

print("\n✓ Full 1,200-intent V2 audit complete")
print("✓ Direction-aware latency semantics applied")
print("✓ All five represented KPI dimensions evaluated")
print("✓ Mobility omission quantified")
print("✓ Empty strict feasible sets preserved")
print("✓ Provisional V1 audit files retained")
print("✓ Frozen source and corpus unchanged")

R2-22 — FULL 1,200-INTENT STRICT V2 FEASIBILITY AUDIT

STRICT V2 OVERALL SUMMARY
------------------------------------------------------------------------------------------------------------------------
                                    metric  count     pct
                              rows_audited   1200 100.000
             legacy_strict_v2_disagreement    229  19.083
legacy_retained_strictly_invalid_candidate    229  19.083
  legacy_rejected_strictly_valid_candidate      0   0.000
              strict_v2_empty_feasible_set     14   1.167
      legacy_nonempty_when_strict_v2_empty     14   1.167
    mobility_contributed_to_overacceptance     61   5.083
                mobility_only_disagreement     37   3.083
     latency_contributed_to_overacceptance     63   5.250
                 latency_only_disagreement     48   4.000

STRICT V2 KPI ATTRIBUTION
------------------------------------------------------------------------------------------------------------------------
            

In [98]:
# ============================================================
# R2-23 — INSPECT AND MAP FROZEN E5 DECISION EVIDENCE
# ============================================================

from pathlib import Path
import pandas as pd
import json
import re

print("=" * 120)
print("R2-23 — INSPECT AND MAP FROZEN E5 DECISION EVIDENCE")
print("=" * 120)

E5_ROOT = (
    BASE_ROOT /
    "02_E5_QWEN37_N100_SEED52"
)

E5_DECISIONS = (
    E5_ROOT /
    "e5_decisions.csv"
)

assert E5_DECISIONS.exists(), (
    f"STOP: missing E5 decision file: {E5_DECISIONS}"
)

e5 = pd.read_csv(E5_DECISIONS)

print("\nE5 DECISION FILE")
print("-" * 120)

print("Path:")
print(E5_DECISIONS)

print("\nSHA256:")
print(file_sha256(E5_DECISIONS))

print("\nShape:")
print(e5.shape)

print("\nColumns:")
print(list(e5.columns))

print("\nDtypes:")
print(e5.dtypes.to_string())

print("\nFIRST 20 DECISIONS")
print("-" * 120)

print(
    e5.head(20).to_string(index=False)
)


# ------------------------------------------------------------
# Unique values / distributions
# ------------------------------------------------------------

print("\nCOLUMN DISTRIBUTIONS")
print("-" * 120)

for col in e5.columns:

    nunique = e5[col].nunique(
        dropna=False
    )

    print(
        f"\n{col} — unique={nunique}"
    )

    if nunique <= 20:

        print(
            e5[col]
            .value_counts(
                dropna=False
            )
            .to_string()
        )


# ------------------------------------------------------------
# Look for direct row identifiers
# ------------------------------------------------------------

possible_id_columns = [
    c for c in e5.columns
    if any(
        token in c.lower()
        for token in [
            "id",
            "idx",
            "index",
            "row",
            "intent",
        ]
    )
]

print("\nPOSSIBLE IDENTIFIER COLUMNS")
print("-" * 120)

print(
    possible_id_columns
    if possible_id_columns
    else "NONE"
)


# ------------------------------------------------------------
# Inspect E5 source for sampling / row-selection logic
# ------------------------------------------------------------

E5_SOURCE = (
    BASE_ROOT /
    "03_SOURCE_CODE" /
    "exp5_agentic.py"
)

assert E5_SOURCE.exists(), (
    f"STOP: missing frozen E5 source: {E5_SOURCE}"
)

print("\nE5 SOURCE")
print("-" * 120)

print(E5_SOURCE)

print("SHA256:")
print(file_sha256(E5_SOURCE))

source_lines = E5_SOURCE.read_text(
    encoding="utf-8"
).splitlines()

search_patterns = [
    "sample",
    "head(",
    "iloc",
    "loc[",
    "random_state",
    "seed",
    "n_intents",
    "corpus",
    "e5_decisions",
    "to_csv",
    "pred",
    "escalated",
]

matches = []

for lineno, line in enumerate(
    source_lines,
    start=1
):

    if any(
        pattern.lower() in line.lower()
        for pattern in search_patterns
    ):

        matches.append(
            (lineno, line)
        )

print("\nE5 SOURCE — RELEVANT LINES")
print("-" * 120)

for lineno, line in matches:

    start = max(
        1,
        lineno - 2
    )

    end = min(
        len(source_lines),
        lineno + 2
    )

    print(
        f"\n--- around line {lineno} ---"
    )

    for i in range(start, end + 1):

        print(
            f"{i:4d}: "
            f"{source_lines[i - 1]}"
        )


# ------------------------------------------------------------
# Inspect run configuration / summary
# ------------------------------------------------------------

summary_path = (
    E5_ROOT /
    "e5_summary.json"
)

if summary_path.exists():

    with open(
        summary_path,
        "r",
        encoding="utf-8"
    ) as f:

        e5_summary = json.load(f)

    print("\nE5 SUMMARY")
    print("-" * 120)

    print(
        json.dumps(
            e5_summary,
            indent=2
        )
    )


# ------------------------------------------------------------
# Check whether decision rows contain text that can be
# directly joined to the frozen corpus.
# ------------------------------------------------------------

text_columns = [
    c for c in e5.columns
    if "text" in c.lower()
    or "prompt" in c.lower()
    or "intent" in c.lower()
]

print("\nPOSSIBLE TEXT/JOIN COLUMNS")
print("-" * 120)

print(
    text_columns
    if text_columns
    else "NONE"
)


# ------------------------------------------------------------
# Save inspection metadata
# ------------------------------------------------------------

inspection = {
    "e5_decisions_path":
        str(E5_DECISIONS),

    "e5_decisions_sha256":
        file_sha256(E5_DECISIONS),

    "e5_rows":
        int(len(e5)),

    "e5_columns":
        list(e5.columns),

    "possible_id_columns":
        possible_id_columns,

    "possible_text_columns":
        text_columns,

    "e5_source_path":
        str(E5_SOURCE),

    "e5_source_sha256":
        file_sha256(E5_SOURCE),

    "mapping_status":
        "PENDING_SOURCE_INSPECTION",
}

inspection_path = (
    AUDIT_DIR /
    "R2_E5_mapping_inspection.json"
)

with open(
    inspection_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        inspection,
        f,
        indent=2
    )

print("\nINSPECTION RECORD SAVED")
print("-" * 120)

print(inspection_path)

print(
    "SHA256:",
    file_sha256(inspection_path)
)


# ------------------------------------------------------------
# Frozen evidence verification
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

assert (
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

print("\n✓ E5 decision evidence inspected")
print("✓ E5 source sampling logic inspected")
print("✓ No row mapping assumed")
print("✓ No predictions regenerated")
print("✓ No E5 experiment rerun")
print("✓ Frozen source and corpus unchanged")

R2-23 — INSPECT AND MAP FROZEN E5 DECISION EVIDENCE

E5 DECISION FILE
------------------------------------------------------------------------------------------------------------------------
Path:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/02_E5_QWEN37_N100_SEED52/e5_decisions.csv

SHA256:
2ae03f26b2c58c3f2e319dcdaa3eb0b91a70d5441c85a41aff4f981270ec0de7

Shape:
(100, 7)

Columns:
['intent', 'verdict', 'retries', 'escalated', 'pred', 'truth', 'uc_kind']

Dtypes:
intent       object
verdict      object
retries       int64
escalated      bool
pred         object
truth        object
uc_kind      object

FIRST 20 DECISIONS
------------------------------------------------------------------------------------------------------------------------
                                                                                                                                                                                     intent  verdict  

In [99]:
# ============================================================
# R2-24 — E5 PREDICTION-LEVEL STRICT V2 SAFETY AUDIT
# ============================================================

import pandas as pd
import numpy as np
import json

print("=" * 120)
print("R2-24 — E5 PREDICTION-LEVEL STRICT V2 SAFETY AUDIT")
print("=" * 120)

RESULTS_RAW = ROOT / "03_RESULTS_RAW"
RESULTS_AGG = ROOT / "04_RESULTS_AGGREGATE"

RESULTS_RAW.mkdir(parents=True, exist_ok=True)
RESULTS_AGG.mkdir(parents=True, exist_ok=True)

assert "e5" in globals()
assert len(e5) == 100

assert "parsed" in globals()
assert len(parsed) == 1200

assert "audit_v2" in globals()
assert len(audit_v2) == 1200


# ------------------------------------------------------------
# 1. Validate exact-text uniqueness in frozen corpus
# ------------------------------------------------------------

corpus_text_counts = (
    parsed["text"]
    .value_counts(dropna=False)
)

duplicate_corpus_texts = corpus_text_counts[
    corpus_text_counts > 1
]

print("\nCORPUS TEXT-KEY CHECK")
print("-" * 120)

print(
    "Frozen corpus rows             :",
    len(parsed)
)

print(
    "Unique intent texts            :",
    parsed["text"].nunique()
)

print(
    "Duplicated intent-text values  :",
    len(duplicate_corpus_texts)
)


# ------------------------------------------------------------
# 2. Validate E5 text uniqueness
# ------------------------------------------------------------

print("\nE5 TEXT-KEY CHECK")
print("-" * 120)

print(
    "E5 decision rows               :",
    len(e5)
)

print(
    "Unique E5 intent texts         :",
    e5["intent"].nunique()
)

assert e5["intent"].nunique() == len(e5), (
    "STOP: duplicate intent text exists inside E5 decisions."
)


# ------------------------------------------------------------
# 3. Exact text mapping
# ------------------------------------------------------------

mapping_base = parsed[
    [
        "id",
        "text",
        "true_slice",
        "uc_kind",
        "latency_ms",
        "throughput_mbps",
        "reliability_pct",
        "density_per_km2",
        "mobility_kmh",
    ]
].copy()

mapped = e5.merge(
    mapping_base,
    left_on="intent",
    right_on="text",
    how="left",
    suffixes=("_e5", "_corpus"),
    indicator=True,
)


print("\nEXACT-TEXT MAPPING")
print("-" * 120)

print(
    mapped["_merge"]
    .value_counts(dropna=False)
    .to_string()
)

unmatched = mapped[
    mapped["_merge"] != "both"
]

print(
    "\nUnmatched E5 decisions:",
    len(unmatched)
)

assert len(unmatched) == 0, (
    "STOP: one or more E5 decisions could not be "
    "mapped exactly to the frozen corpus."
)

assert len(mapped) == 100


# ------------------------------------------------------------
# 4. Validate truth and subgroup after mapping
# ------------------------------------------------------------

mapped["truth_match"] = (
    mapped["truth"]
    == mapped["true_slice"]
)

mapped["uc_kind_match"] = (
    mapped["uc_kind_e5"]
    == mapped["uc_kind_corpus"]
)

print("\nMAPPING CONSISTENCY")
print("-" * 120)

print(
    "Truth matches     :",
    int(mapped["truth_match"].sum()),
    "/",
    len(mapped)
)

print(
    "uc_kind matches   :",
    int(mapped["uc_kind_match"].sum()),
    "/",
    len(mapped)
)

assert mapped["truth_match"].all(), (
    "STOP: E5 truth labels do not exactly match "
    "the mapped frozen corpus."
)

assert mapped["uc_kind_match"].all(), (
    "STOP: E5 uc_kind does not exactly match "
    "the mapped frozen corpus."
)


# ------------------------------------------------------------
# 5. Compute legacy and strict validity of ACTUAL prediction
# ------------------------------------------------------------

legacy_sets = []
strict_sets = []

legacy_valid = []
strict_valid = []

for _, row in mapped.iterrows():

    legacy = frozen.feasible_set(row)
    strict = strict_feasible_set_v2(row)

    legacy_sets.append(
        "|".join(legacy)
    )

    strict_sets.append(
        "|".join(strict)
    )

    legacy_valid.append(
        row["pred"] in legacy
    )

    strict_valid.append(
        row["pred"] in strict
    )


mapped["legacy_feasible_set"] = legacy_sets
mapped["strict_v2_feasible_set"] = strict_sets

mapped["legacy_prediction_valid"] = legacy_valid
mapped["strict_v2_prediction_valid"] = strict_valid


# ------------------------------------------------------------
# 6. Decision safety categories
# ------------------------------------------------------------

def safety_category(row):

    legacy_ok = bool(
        row["legacy_prediction_valid"]
    )

    strict_ok = bool(
        row["strict_v2_prediction_valid"]
    )

    if legacy_ok and strict_ok:
        return "legacy_valid_strict_valid"

    if legacy_ok and not strict_ok:
        return "legacy_valid_strict_invalid"

    if not legacy_ok and strict_ok:
        return "legacy_invalid_strict_valid"

    return "legacy_invalid_strict_invalid"


mapped["safety_category"] = mapped.apply(
    safety_category,
    axis=1
)

# This is the critical prediction-level event.
mapped["false_feasible_decision"] = (
    mapped["legacy_prediction_valid"]
    &
    ~mapped["strict_v2_prediction_valid"]
)


# ------------------------------------------------------------
# 7. Strict-empty request indicator
# ------------------------------------------------------------

mapped["strict_v2_empty_request"] = (
    mapped["strict_v2_feasible_set"] == ""
)


# ------------------------------------------------------------
# 8. Prediction correctness
# ------------------------------------------------------------

mapped["prediction_correct"] = (
    mapped["pred"]
    == mapped["truth"]
)


# ------------------------------------------------------------
# 9. Summaries
# ------------------------------------------------------------

N = len(mapped)

n_legacy_valid = int(
    mapped[
        "legacy_prediction_valid"
    ].sum()
)

n_strict_valid = int(
    mapped[
        "strict_v2_prediction_valid"
    ].sum()
)

n_false_feasible = int(
    mapped[
        "false_feasible_decision"
    ].sum()
)

n_strict_invalid = int(
    (
        ~mapped[
            "strict_v2_prediction_valid"
        ]
    ).sum()
)

n_empty_requests = int(
    mapped[
        "strict_v2_empty_request"
    ].sum()
)

n_false_feasible_accept = int(
    (
        mapped["false_feasible_decision"]
        &
        (mapped["verdict"] == "ACCEPT")
    ).sum()
)

n_false_feasible_escalate = int(
    (
        mapped["false_feasible_decision"]
        &
        (mapped["verdict"] == "ESCALATE")
    ).sum()
)


summary = pd.DataFrame([

    {
        "metric":
            "e5_decisions_audited",
        "count":
            N,
        "pct":
            100.0,
    },

    {
        "metric":
            "legacy_prediction_valid",
        "count":
            n_legacy_valid,
        "pct":
            100.0 * n_legacy_valid / N,
    },

    {
        "metric":
            "strict_v2_prediction_valid",
        "count":
            n_strict_valid,
        "pct":
            100.0 * n_strict_valid / N,
    },

    {
        "metric":
            "strict_v2_prediction_invalid",
        "count":
            n_strict_invalid,
        "pct":
            100.0 * n_strict_invalid / N,
    },

    {
        "metric":
            "false_feasible_decision",
        "count":
            n_false_feasible,
        "pct":
            100.0 * n_false_feasible / N,
    },

    {
        "metric":
            "strict_v2_empty_request",
        "count":
            n_empty_requests,
        "pct":
            100.0 * n_empty_requests / N,
    },

    {
        "metric":
            "false_feasible_and_ACCEPT",
        "count":
            n_false_feasible_accept,
        "pct":
            100.0 * n_false_feasible_accept / N,
    },

    {
        "metric":
            "false_feasible_and_ESCALATE",
        "count":
            n_false_feasible_escalate,
        "pct":
            100.0 * n_false_feasible_escalate / N,
    },
])


# ------------------------------------------------------------
# 10. Safety matrix
# ------------------------------------------------------------

matrix = (
    mapped[
        "safety_category"
    ]
    .value_counts()
    .rename_axis(
        "safety_category"
    )
    .reset_index(
        name="count"
    )
)

matrix["pct"] = (
    100.0
    * matrix["count"]
    / N
)


# ------------------------------------------------------------
# 11. Print main results
# ------------------------------------------------------------

print("\nPREDICTION-LEVEL SAFETY SUMMARY")
print("-" * 120)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nLEGACY VS STRICT V2 DECISION MATRIX")
print("-" * 120)

print(
    matrix.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ------------------------------------------------------------
# 12. Critical false-feasible cases
# ------------------------------------------------------------

false_cases = mapped[
    mapped[
        "false_feasible_decision"
    ]
].copy()

print("\nFALSE-FEASIBLE DECISIONS")
print("-" * 120)

if len(false_cases):

    print(
        false_cases[
            [
                "id",
                "pred",
                "truth",
                "verdict",
                "escalated",
                "uc_kind_e5",
                "legacy_feasible_set",
                "strict_v2_feasible_set",
                "latency_ms",
                "throughput_mbps",
                "reliability_pct",
                "density_per_km2",
                "mobility_kmh",
            ]
        ]
        .to_string(index=False)
    )

else:
    print("NONE")


# ------------------------------------------------------------
# 13. Strict-empty requests among E5 decisions
# ------------------------------------------------------------

empty_cases = mapped[
    mapped[
        "strict_v2_empty_request"
    ]
].copy()

print("\nSTRICT-EMPTY REQUESTS IN E5 SAMPLE")
print("-" * 120)

if len(empty_cases):

    print(
        empty_cases[
            [
                "id",
                "pred",
                "truth",
                "verdict",
                "escalated",
                "legacy_feasible_set",
                "strict_v2_feasible_set",
            ]
        ]
        .to_string(index=False)
    )

else:
    print("NONE")


# ------------------------------------------------------------
# 14. Important integrity check against frozen summary
# ------------------------------------------------------------

print("\nLEGACY CONTRACT CONSISTENCY")
print("-" * 120)

print(
    "Recomputed legacy-valid decisions:",
    n_legacy_valid,
    "/",
    N
)

assert n_legacy_valid == 100, (
    "STOP: recomputed legacy validity does not match "
    "the frozen E5 contract_valid=1.0 evidence."
)

print(
    "✓ Recomputed legacy validity matches "
    "frozen E5 contract_valid = 1.0"
)


# ------------------------------------------------------------
# 15. Save evidence
# ------------------------------------------------------------

audit_path = (
    RESULTS_RAW /
    "R2_E5_prediction_level_audit_v2.csv"
)

false_path = (
    RESULTS_RAW /
    "R2_E5_false_feasible_decisions_v2.csv"
)

empty_path = (
    RESULTS_RAW /
    "R2_E5_strict_empty_requests_v2.csv"
)

summary_path = (
    RESULTS_AGG /
    "R2_E5_prediction_safety_summary_v2.csv"
)

matrix_path = (
    RESULTS_AGG /
    "R2_E5_legacy_vs_strict_matrix_v2.csv"
)


mapped.to_csv(
    audit_path,
    index=False
)

false_cases.to_csv(
    false_path,
    index=False
)

empty_cases.to_csv(
    empty_path,
    index=False
)

summary.to_csv(
    summary_path,
    index=False
)

matrix.to_csv(
    matrix_path,
    index=False
)


# ------------------------------------------------------------
# 16. Mapping provenance
# ------------------------------------------------------------

mapping_provenance = {

    "mapping_key":
        "exact frozen intent text",

    "e5_rows":
        int(len(e5)),

    "mapped_rows":
        int(len(mapped)),

    "truth_matches":
        int(mapped["truth_match"].sum()),

    "uc_kind_matches":
        int(mapped["uc_kind_match"].sum()),

    "predictions_regenerated":
        False,

    "e5_rerun":
        False,

    "strict_oracle":
        "R2_strict_feasibility_oracle_v2.py",

    "interpretation":
        (
            "Each frozen E5 decision was joined to the "
            "frozen 1200-row corpus by exact intent text. "
            "The original E5 prediction was then audited "
            "against both legacy and strict V2 feasibility."
        ),
}

mapping_path = (
    AUDIT_DIR /
    "R2_E5_exact_text_mapping_provenance.json"
)

with open(
    mapping_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        mapping_provenance,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 17. Hash outputs
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    audit_path,
    false_path,
    empty_path,
    summary_path,
    matrix_path,
    mapping_path,
]:

    print(p)
    print(
        "SHA256:",
        file_sha256(p)
    )


# ------------------------------------------------------------
# 18. Frozen evidence verification
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

EXPECTED_E5_HASH = (
    "2ae03f26b2c58c3f2e319dcdaa3eb0b"
    "91a70d5441c85a41aff4f981270ec0de7"
)

assert (
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

assert (
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

assert (
    file_sha256(E5_DECISIONS)
    == EXPECTED_E5_HASH
)

print("\n✓ All 100 E5 decisions mapped to frozen corpus")
print("✓ Original predictions preserved")
print("✓ Legacy validity independently reproduced")
print("✓ Strict V2 prediction validity calculated")
print("✓ False-feasible decisions isolated")
print("✓ No model retraining")
print("✓ No E5 rerun")
print("✓ Frozen evidence unchanged")

R2-24 — E5 PREDICTION-LEVEL STRICT V2 SAFETY AUDIT

CORPUS TEXT-KEY CHECK
------------------------------------------------------------------------------------------------------------------------
Frozen corpus rows             : 1200
Unique intent texts            : 1200
Duplicated intent-text values  : 0

E5 TEXT-KEY CHECK
------------------------------------------------------------------------------------------------------------------------
E5 decision rows               : 100
Unique E5 intent texts         : 100

EXACT-TEXT MAPPING
------------------------------------------------------------------------------------------------------------------------
_merge
both          100
left_only       0
right_only      0

Unmatched E5 decisions: 0

MAPPING CONSISTENCY
------------------------------------------------------------------------------------------------------------------------
Truth matches     : 100 / 100
uc_kind matches   : 100 / 100

PREDICTION-LEVEL SAFETY SUMMARY
----------------

In [100]:
# ============================================================
# R2-25 — PUBLICATION-READY R2 RESULTS + FINDINGS
# ============================================================

import pandas as pd
import json
from pathlib import Path

print("=" * 120)
print("R2-25 — PUBLICATION-READY STRICT FEASIBILITY RESULTS")
print("=" * 120)

DOC_DIR = ROOT / "06_DOCUMENTATION"
AGG_DIR = ROOT / "04_RESULTS_AGGREGATE"

DOC_DIR.mkdir(parents=True, exist_ok=True)
AGG_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# TABLE 1 — REQUEST/CANDIDATE-LEVEL AUDIT
# ------------------------------------------------------------

publication_request = pd.DataFrame([
    {
        "Measure":
            "Frozen intents audited",
        "Count":
            1200,
        "Percent":
            100.000,
    },
    {
        "Measure":
            "Legacy vs strict candidate-set disagreement",
        "Count":
            229,
        "Percent":
            19.083,
    },
    {
        "Measure":
            "Legacy retained >=1 strictly invalid candidate",
        "Count":
            229,
        "Percent":
            19.083,
    },
    {
        "Measure":
            "Legacy rejected a strictly valid candidate",
        "Count":
            0,
        "Percent":
            0.000,
    },
    {
        "Measure":
            "Strict empty feasible set",
        "Count":
            14,
        "Percent":
            1.167,
    },
    {
        "Measure":
            "Legacy non-empty when strict set empty",
        "Count":
            14,
        "Percent":
            1.167,
    },
    {
        "Measure":
            "Mobility contributed to over-acceptance",
        "Count":
            61,
        "Percent":
            5.083,
    },
    {
        "Measure":
            "Mobility-only disagreement",
        "Count":
            37,
        "Percent":
            3.083,
    },
])


# ------------------------------------------------------------
# TABLE 2 — KPI ATTRIBUTION
# ------------------------------------------------------------

publication_kpi = pd.DataFrame([
    {
        "KPI":
            "Latency",
        "Rows":
            63,
        "Percent_of_1200":
            5.250,
    },
    {
        "KPI":
            "Throughput",
        "Rows":
            37,
        "Percent_of_1200":
            3.083,
    },
    {
        "KPI":
            "Reliability",
        "Rows":
            0,
        "Percent_of_1200":
            0.000,
    },
    {
        "KPI":
            "Density",
        "Rows":
            103,
        "Percent_of_1200":
            8.583,
    },
    {
        "KPI":
            "Mobility",
        "Rows":
            61,
        "Percent_of_1200":
            5.083,
    },
])


# ------------------------------------------------------------
# TABLE 3 — E5 PREDICTION-LEVEL AUDIT
# ------------------------------------------------------------

publication_e5 = pd.DataFrame([
    {
        "Measure":
            "Frozen E5 decisions audited",
        "Count":
            100,
        "Percent":
            100.000,
    },
    {
        "Measure":
            "Legacy-valid predictions",
        "Count":
            100,
        "Percent":
            100.000,
    },
    {
        "Measure":
            "Strict-valid predictions",
        "Count":
            97,
        "Percent":
            97.000,
    },
    {
        "Measure":
            "False-feasible predictions",
        "Count":
            3,
        "Percent":
            3.000,
    },
    {
        "Measure":
            "False-feasible + ACCEPT",
        "Count":
            2,
        "Percent":
            2.000,
    },
    {
        "Measure":
            "False-feasible + ESCALATE",
        "Count":
            1,
        "Percent":
            1.000,
    },
])


# ------------------------------------------------------------
# TABLE 4 — CRITICAL CASES
# ------------------------------------------------------------

critical_cases = false_cases[
    [
        "id",
        "pred",
        "truth",
        "verdict",
        "escalated",
        "latency_ms",
        "throughput_mbps",
        "reliability_pct",
        "density_per_km2",
        "mobility_kmh",
    ]
].copy()

critical_cases["classification_correct"] = (
    critical_cases["pred"]
    ==
    critical_cases["truth"]
)

# Add reason from full V2 request audit.
reason_lookup = audit_v2[
    [
        "id",
        "extra_violation_kpis",
    ]
].copy()

critical_cases = critical_cases.merge(
    reason_lookup,
    on="id",
    how="left"
)


# ------------------------------------------------------------
# Save publication tables
# ------------------------------------------------------------

request_path = (
    AGG_DIR /
    "R2_publication_request_level_table.csv"
)

kpi_path = (
    AGG_DIR /
    "R2_publication_kpi_table.csv"
)

e5_path = (
    AGG_DIR /
    "R2_publication_E5_decision_table.csv"
)

critical_path = (
    AGG_DIR /
    "R2_publication_critical_cases.csv"
)

publication_request.to_csv(
    request_path,
    index=False
)

publication_kpi.to_csv(
    kpi_path,
    index=False
)

publication_e5.to_csv(
    e5_path,
    index=False
)

critical_cases.to_csv(
    critical_path,
    index=False
)


# ------------------------------------------------------------
# Publication-ready findings text
# ------------------------------------------------------------

findings = """# R2 — Strict Feasibility and Validity Audit

## Status

R2 evaluates the feasibility and safety implications of the original
legacy contract used in the frozen E1-E5 experiments.

The original E1-E5 evidence was not modified or regenerated.
R2 is an independent post-hoc strict re-analysis of the frozen evidence.

The authoritative strict oracle is:

R2_strict_feasibility_oracle_v2.py

The earlier V1 strict audit is retained only for provenance and is
superseded by V2 because V2 applies direction-aware latency semantics.


## Strict feasibility semantics

Five KPI dimensions represented by the experiment are evaluated:

- latency
- throughput
- reliability
- device density
- mobility

Latency is interpreted as an upper-bound SLA. A slice is latency-feasible
when its minimum supported latency is less than or equal to the requested
maximum latency.

Throughput, reliability, density, and mobility are capability requirements.
A slice is feasible when the requested requirement does not exceed the
slice's maximum supported capability.

Missing KPI values are treated as unconstrained.

Unlike the legacy implementation, the strict oracle preserves an empty
feasible set when no catalogue slice satisfies the complete request.


## Request-level results

The strict V2 oracle was applied to all 1,200 frozen intents using KPI slots
reconstructed by the frozen parser.

Legacy and strict candidate sets disagreed for 229 of 1,200 intents
(19.08%).

All 229 disagreements involved legacy over-acceptance: the legacy contract
retained at least one candidate that failed the stricter five-KPI audit.

There were no cases in which the legacy contract rejected a candidate that
the strict V2 oracle considered feasible.

Fourteen intents (1.17%) had no feasible slice under the strict oracle.
Nevertheless, the legacy implementation returned a non-empty candidate set
for all fourteen because its empty-set fallback restores the full catalogue.

This demonstrates that the legacy contract can conceal infeasible requests
instead of explicitly representing infeasibility.


## KPI attribution

Density contributed to legacy over-acceptance in 103 of 1,200 intents
(8.58%), making it the largest observed contributor.

Latency contributed in 63 intents (5.25%).

Mobility contributed in 61 intents (5.08%).

Throughput contributed in 37 intents (3.08%).

Reliability produced no disagreement in this frozen corpus.

The attribution categories can overlap because a single request may violate
multiple KPI constraints.


## Mobility finding

Mobility is represented in the generated intents and catalogue but was not
checked by the legacy feasibility function.

The strict audit identified 61 intents (5.08%) in which mobility contributed
to legacy over-acceptance.

Thirty-seven intents (3.08%) were mobility-only disagreements, meaning the
legacy-extra candidate or candidates failed solely because of mobility.

This directly quantifies the impact of the missing mobility check.


## Prediction-level E5 audit

The 100 frozen E5 decisions were mapped back to the frozen 1,200-intent
corpus using exact intent text.

All 100 mappings were exact.

Truth labels matched for 100/100 rows and use-case categories matched for
100/100 rows.

The original predictions were not regenerated.

The original legacy contract classified all 100 E5 predictions as valid,
reproducing the frozen E5 contract-validity result of 100%.

Under strict V2 feasibility, 97 of the 100 predictions remained feasible.

Three predictions (3.0%) were legacy-valid but strict-invalid and are
therefore classified as false-feasible decisions under the strict audit.


## Operational safety finding

Of the three false-feasible E5 decisions:

- one was already escalated by the agent;
- two were autonomously ACCEPTed.

Thus, 2% of the audited E5 decisions were strict-infeasible yet autonomously
accepted under the original pipeline.

All three false-feasible predictions occurred on requests for which the
strict feasible set was empty.

Therefore these cases cannot be corrected merely by choosing another
catalogue slice. The safe outcome is escalation, rejection, or explicit
infeasibility handling.


## Classification accuracy versus feasibility

Two of the three false-feasible cases had predictions equal to their
ground-truth slice labels.

This demonstrates an important distinction between classification
correctness and operational feasibility.

A model may correctly predict the dataset's nominal slice label while the
complete KPI combination is infeasible under the catalogue constraints.

Consequently, classification accuracy and macro-F1 alone are insufficient
safety indicators for autonomous slice orchestration.


## Interpretation

R2 does not invalidate the original E1-E5 experimental results.

Instead, it identifies a limitation in the original contract-validity
definition and quantifies its effect using frozen experimental evidence.

The original results should therefore be reported as legacy-contract
results, while the R2 strict audit should be reported separately as a
safety-oriented re-analysis.

The results motivate explicit strict feasibility enforcement before
autonomous slice deployment.


## Scope limitation

Row-level predictions were preserved for the 100-intent E5 experiment,
allowing direct strict prediction-level re-analysis.

Equivalent row-level predictions were not preserved for every E1-E4
multi-seed experiment. Therefore prediction-level strict feasibility metrics
should not be retrospectively claimed for those runs without rerunning the
models.

The 1,200-intent request-level audit and the 100-intent E5 prediction-level
audit should therefore be reported as distinct analyses.


## Main quantitative findings

1. Legacy/strict candidate disagreement: 229/1,200 (19.08%).
2. Strict-empty requests: 14/1,200 (1.17%).
3. Legacy returned candidates for all 14 strict-empty requests.
4. Mobility contributed to 61/1,200 disagreements (5.08%).
5. Mobility-only disagreements: 37/1,200 (3.08%).
6. E5 legacy contract validity: 100/100 (100%).
7. E5 strict V2 prediction validity: 97/100 (97%).
8. E5 false-feasible predictions: 3/100 (3%).
9. False-feasible autonomous ACCEPT decisions: 2/100 (2%).
10. Two false-feasible decisions were classification-correct but
    operationally infeasible.
"""


findings_path = (
    DOC_DIR /
    "R2_FINAL_FINDINGS.md"
)

findings_path.write_text(
    findings,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Methodology note
# ------------------------------------------------------------

methodology = """# R2 Methodology — Strict Feasibility Re-analysis

## Objective

R2 audits whether the original network-slice feasibility contract can admit
candidate slices that violate the represented KPI requirements.

## Evidence preservation

The E1-E5 source code, corpus, predictions, and result artifacts were treated
as frozen evidence.

No model was retrained.

No E5 prediction was regenerated.

No original result was overwritten.

## Parser reconstruction

The frozen corpus stores natural-language intent text and generated KPI truth
values.

Because the original experiment did not persist all parsed KPI slots at
row level, R2 reconstructs those slots deterministically using the original
frozen parser.

Generated truth values are not substituted for parsed values because the
runtime feasibility contract operates on parsed intent values.

## Strict oracle

The independent V2 oracle evaluates all five represented KPI dimensions:
latency, throughput, reliability, density, and mobility.

Missing KPI values are treated as unconstrained.

Latency is evaluated as an upper-bound SLA:

    catalogue minimum latency <= requested maximum latency

Other KPI requirements are evaluated as required capabilities:

    requested requirement <= catalogue maximum capability

An empty feasible set is preserved.

## Legacy comparison

For each frozen intent, the candidate set returned by the original legacy
feasibility function is compared with the strict V2 candidate set.

Legacy-extra candidates are candidates accepted by the legacy function but
rejected by the strict oracle.

Legacy-missing candidates are candidates rejected by the legacy function but
accepted by the strict oracle.

## E5 prediction-level audit

The frozen E5 decision artifact contains the original intent text, prediction,
truth label, verdict, escalation state, and subgroup.

Each E5 decision is mapped to the frozen corpus by exact intent text.

Mapping integrity is verified against both truth label and subgroup.

The original prediction is then tested against:

1. the original legacy feasible set; and
2. the independent strict V2 feasible set.

A false-feasible decision is defined as:

    legacy prediction valid = True
    AND
    strict V2 prediction valid = False

No prediction is regenerated during this analysis.

## Reporting

Candidate-set disagreement and prediction-level false-feasibility are
reported separately.

Candidate-set disagreement must not be described as a false-feasible
prediction unless an actual selected prediction is available.

This distinction prevents overstatement of the R2 findings.
"""


methodology_path = (
    DOC_DIR /
    "R2_METHODOLOGY.md"
)

methodology_path.write_text(
    methodology,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Print publication tables
# ------------------------------------------------------------

print("\nTABLE 1 — REQUEST-LEVEL AUDIT")
print("-" * 120)

print(
    publication_request.to_string(
        index=False
    )
)

print("\nTABLE 2 — KPI ATTRIBUTION")
print("-" * 120)

print(
    publication_kpi.to_string(
        index=False
    )
)

print("\nTABLE 3 — E5 DECISION SAFETY")
print("-" * 120)

print(
    publication_e5.to_string(
        index=False
    )
)

print("\nTABLE 4 — CRITICAL FALSE-FEASIBLE CASES")
print("-" * 120)

print(
    critical_cases.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Hash outputs
# ------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 120)

for p in [
    request_path,
    kpi_path,
    e5_path,
    critical_path,
    findings_path,
    methodology_path,
]:

    print(p)
    print(
        "SHA256:",
        file_sha256(p)
    )


print("\n✓ Publication-ready R2 tables created")
print("✓ Final R2 findings documented")
print("✓ R2 methodology documented")
print("✓ Candidate-level and decision-level claims separated")
print("✓ R2 V2 is authoritative")
print("✓ R2 V1 retained only as provenance")

R2-25 — PUBLICATION-READY STRICT FEASIBILITY RESULTS

TABLE 1 — REQUEST-LEVEL AUDIT
------------------------------------------------------------------------------------------------------------------------
                                       Measure  Count  Percent
                        Frozen intents audited   1200  100.000
   Legacy vs strict candidate-set disagreement    229   19.083
Legacy retained >=1 strictly invalid candidate    229   19.083
    Legacy rejected a strictly valid candidate      0    0.000
                     Strict empty feasible set     14    1.167
        Legacy non-empty when strict set empty     14    1.167
       Mobility contributed to over-acceptance     61    5.083
                    Mobility-only disagreement     37    3.083

TABLE 2 — KPI ATTRIBUTION
------------------------------------------------------------------------------------------------------------------------
        KPI  Rows  Percent_of_1200
    Latency    63            5.250
 Throughpu

In [101]:
# ============================================================
# R2-26 — FINAL R2 INTEGRITY AND CONSISTENCY VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import json

print("=" * 120)
print("R2-26 — FINAL R2 INTEGRITY AND CONSISTENCY VALIDATION")
print("=" * 120)

# ------------------------------------------------------------
# Directories
# ------------------------------------------------------------

AUDIT_DIR = ROOT / "02_R2_AUDIT"
RAW_DIR   = ROOT / "03_RESULTS_RAW"
AGG_DIR   = ROOT / "04_RESULTS_AGGREGATE"
DOC_DIR   = ROOT / "06_DOCUMENTATION"

# ------------------------------------------------------------
# Required authoritative V2 artifacts
# ------------------------------------------------------------

required = [

    # Oracle / audit
    AUDIT_DIR / "R2_strict_feasibility_oracle_v2.py",
    AUDIT_DIR / "R2_legacy_kpi_coverage.csv",
    AUDIT_DIR / "R2_parser_truth_consistency.csv",
    AUDIT_DIR / "R2_parsed_slot_availability.csv",
    AUDIT_DIR / "R2_parsed_slot_reconstruction_provenance.json",
    AUDIT_DIR / "R2_E5_mapping_inspection.json",
    AUDIT_DIR / "R2_E5_exact_text_mapping_provenance.json",

    # Raw evidence
    RAW_DIR / "R2_reconstructed_parsed_slots.csv",
    RAW_DIR / "R2_controlled_feasibility_cases_v2.csv",
    RAW_DIR / "R2_per_intent_feasibility_audit_v2.csv",
    RAW_DIR / "R2_strict_empty_cases_v2.csv",
    RAW_DIR / "R2_mobility_only_disagreements_v2.csv",
    RAW_DIR / "R2_E5_prediction_level_audit_v2.csv",
    RAW_DIR / "R2_E5_false_feasible_decisions_v2.csv",
    RAW_DIR / "R2_E5_strict_empty_requests_v2.csv",

    # Aggregate evidence
    AGG_DIR / "R2_feasibility_summary_v2.csv",
    AGG_DIR / "R2_kpi_attribution_v2.csv",
    AGG_DIR / "R2_E5_prediction_safety_summary_v2.csv",
    AGG_DIR / "R2_E5_legacy_vs_strict_matrix_v2.csv",
    AGG_DIR / "R2_publication_request_level_table.csv",
    AGG_DIR / "R2_publication_kpi_table.csv",
    AGG_DIR / "R2_publication_E5_decision_table.csv",
    AGG_DIR / "R2_publication_critical_cases.csv",

    # Documentation
    DOC_DIR / "R2_FINAL_FINDINGS.md",
    DOC_DIR / "R2_METHODOLOGY.md",
]

print("\nREQUIRED ARTIFACT CHECK")
print("-" * 120)

missing = []

for p in required:

    status = "PASS" if p.exists() else "MISSING"

    print(
        f"{status:8s} "
        f"{p.relative_to(ROOT)}"
    )

    if not p.exists():
        missing.append(p)

assert not missing, (
    "STOP: required R2 artifacts are missing."
)

print(
    f"\n✓ All {len(required)} required "
    "authoritative artifacts exist"
)


# ------------------------------------------------------------
# Load authoritative results
# ------------------------------------------------------------

request_audit = pd.read_csv(
    RAW_DIR /
    "R2_per_intent_feasibility_audit_v2.csv"
)

controlled = pd.read_csv(
    RAW_DIR /
    "R2_controlled_feasibility_cases_v2.csv"
)

empty_cases = pd.read_csv(
    RAW_DIR /
    "R2_strict_empty_cases_v2.csv"
)

mobility_only = pd.read_csv(
    RAW_DIR /
    "R2_mobility_only_disagreements_v2.csv"
)

e5_audit = pd.read_csv(
    RAW_DIR /
    "R2_E5_prediction_level_audit_v2.csv"
)

e5_false = pd.read_csv(
    RAW_DIR /
    "R2_E5_false_feasible_decisions_v2.csv"
)

e5_empty = pd.read_csv(
    RAW_DIR /
    "R2_E5_strict_empty_requests_v2.csv"
)

publication_request = pd.read_csv(
    AGG_DIR /
    "R2_publication_request_level_table.csv"
)

publication_e5 = pd.read_csv(
    AGG_DIR /
    "R2_publication_E5_decision_table.csv"
)

critical = pd.read_csv(
    AGG_DIR /
    "R2_publication_critical_cases.csv"
)


# ------------------------------------------------------------
# Structural assertions
# ------------------------------------------------------------

print("\nSTRUCTURAL CONSISTENCY")
print("-" * 120)

checks = []

def check(name, condition):

    condition = bool(condition)

    checks.append(
        {
            "check": name,
            "passed": condition,
        }
    )

    print(
        f"{'PASS' if condition else 'FAIL':5s} "
        f"{name}"
    )

    assert condition, (
        f"STOP: integrity check failed: {name}"
    )


check(
    "controlled V2 suite = 16 rows",
    len(controlled) == 16
)

check(
    "controlled V2 suite = 16/16 PASS",
    controlled["strict_test_passed"].sum() == 16
)

check(
    "request-level audit = 1200 rows",
    len(request_audit) == 1200
)

check(
    "legacy/strict disagreements = 229",
    request_audit[
        "legacy_strict_v2_disagreement"
    ].sum() == 229
)

check(
    "legacy over-acceptance rows = 229",
    (
        request_audit[
            "legacy_extra_count"
        ] > 0
    ).sum() == 229
)

check(
    "legacy missing candidate rows = 0",
    (
        request_audit[
            "legacy_missing_count"
        ] > 0
    ).sum() == 0
)

check(
    "strict-empty requests = 14",
    request_audit[
        "strict_v2_empty"
    ].sum() == 14
)

check(
    "strict-empty artifact = 14 rows",
    len(empty_cases) == 14
)

check(
    "legacy nonempty on all strict-empty requests",
    request_audit[
        "legacy_nonempty_strict_empty"
    ].sum() == 14
)

check(
    "mobility contribution = 61 rows",
    request_audit[
        "mobility_contributed"
    ].sum() == 61
)

check(
    "mobility-only disagreements = 37",
    request_audit[
        "mobility_only_disagreement"
    ].sum() == 37
)

check(
    "mobility-only artifact = 37 rows",
    len(mobility_only) == 37
)


# ------------------------------------------------------------
# E5 assertions
# ------------------------------------------------------------

check(
    "E5 prediction audit = 100 rows",
    len(e5_audit) == 100
)

check(
    "E5 legacy-valid = 100",
    e5_audit[
        "legacy_prediction_valid"
    ].sum() == 100
)

check(
    "E5 strict-valid = 97",
    e5_audit[
        "strict_v2_prediction_valid"
    ].sum() == 97
)

check(
    "E5 false-feasible = 3",
    e5_audit[
        "false_feasible_decision"
    ].sum() == 3
)

check(
    "false-feasible artifact = 3 rows",
    len(e5_false) == 3
)

check(
    "E5 strict-empty = 3",
    e5_audit[
        "strict_v2_empty_request"
    ].sum() == 3
)

check(
    "E5 strict-empty artifact = 3 rows",
    len(e5_empty) == 3
)

check(
    "false-feasible ACCEPT = 2",
    (
        e5_audit[
            "false_feasible_decision"
        ]
        &
        (
            e5_audit[
                "verdict"
            ] == "ACCEPT"
        )
    ).sum() == 2
)

check(
    "false-feasible ESCALATE = 1",
    (
        e5_audit[
            "false_feasible_decision"
        ]
        &
        (
            e5_audit[
                "verdict"
            ] == "ESCALATE"
        )
    ).sum() == 1
)

check(
    "critical publication table = 3 rows",
    len(critical) == 3
)

check(
    "classification-correct false-feasible = 2",
    critical[
        "classification_correct"
    ].sum() == 2
)


# ------------------------------------------------------------
# Frozen source integrity
# ------------------------------------------------------------

EXPECTED_SOURCE_HASH = (
    "7bf2f1190a2928f273578425f194fc327"
    "2385d4baaa9044bff771d0fd748ae05"
)

EXPECTED_CORPUS_HASH = (
    "5358e25a06c863fcb25b4d906608783d"
    "612c0b1411373f140573408555b04b08"
)

EXPECTED_E5_HASH = (
    "2ae03f26b2c58c3f2e319dcdaa3eb0b"
    "91a70d5441c85a41aff4f981270ec0de7"
)

check(
    "frozen xagentic_slice.py unchanged",
    sha256_file(SLICE_SOURCE)
    == EXPECTED_SOURCE_HASH
)

check(
    "frozen 1200-row corpus unchanged",
    file_sha256(CORPUS_PATH)
    == EXPECTED_CORPUS_HASH
)

check(
    "frozen E5 decisions unchanged",
    file_sha256(E5_DECISIONS)
    == EXPECTED_E5_HASH
)


# ------------------------------------------------------------
# V1/V2 provenance check
# ------------------------------------------------------------

print("\nV1 / V2 PROVENANCE")
print("-" * 120)

V1 = (
    AUDIT_DIR /
    "R2_strict_feasibility_oracle.py"
)

V2 = (
    AUDIT_DIR /
    "R2_strict_feasibility_oracle_v2.py"
)

check(
    "V1 oracle retained",
    V1.exists()
)

check(
    "V2 oracle retained",
    V2.exists()
)

check(
    "V1 and V2 are distinct files",
    file_sha256(V1)
    != file_sha256(V2)
)

print(
    "\nV1 SHA256:",
    file_sha256(V1)
)

print(
    "V2 SHA256:",
    file_sha256(V2)
)


# ------------------------------------------------------------
# Findings / methodology checks
# ------------------------------------------------------------

findings_text = (
    DOC_DIR /
    "R2_FINAL_FINDINGS.md"
).read_text(
    encoding="utf-8"
)

method_text = (
    DOC_DIR /
    "R2_METHODOLOGY.md"
).read_text(
    encoding="utf-8"
)

check(
    "findings contain 229/1,200",
    "229/1,200" in findings_text
)

check(
    "findings contain 14/1,200",
    "14/1,200" in findings_text
)

check(
    "findings contain 97/100",
    "97/100" in findings_text
)

check(
    "findings contain 3/100",
    "3/100" in findings_text
)

check(
    "findings contain 2/100",
    "2/100" in findings_text
)

check(
    "methodology defines false-feasible",
    "false-feasible decision" in method_text.lower()
)


# ------------------------------------------------------------
# Save validation report
# ------------------------------------------------------------

checks_df = pd.DataFrame(checks)

validation_path = (
    AGG_DIR /
    "R2_FINAL_INTEGRITY_CHECKS.csv"
)

checks_df.to_csv(
    validation_path,
    index=False
)


# ------------------------------------------------------------
# Closure summary
# ------------------------------------------------------------

closure = {

    "r2_status":
        "INTEGRITY_VALIDATED",

    "authoritative_oracle":
        "R2_strict_feasibility_oracle_v2.py",

    "v1_status":
        "SUPERSEDED_RETAINED_FOR_PROVENANCE",

    "controlled_tests":
        {
            "passed": 16,
            "total": 16,
        },

    "request_level":
        {
            "rows": 1200,
            "legacy_strict_disagreement": 229,
            "strict_empty": 14,
            "mobility_contributed": 61,
            "mobility_only": 37,
        },

    "e5_decision_level":
        {
            "rows": 100,
            "legacy_valid": 100,
            "strict_valid": 97,
            "false_feasible": 3,
            "false_feasible_accept": 2,
            "false_feasible_escalate": 1,
        },

    "frozen_evidence_unchanged":
        True,

    "model_rerun":
        False,

    "model_retraining":
        False,

    "ready_for_packaging":
        True,
}

closure_path = (
    DOC_DIR /
    "R2_CLOSURE.json"
)

with open(
    closure_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        closure,
        f,
        indent=2
    )


print("\nVALIDATION ARTIFACTS")
print("-" * 120)

print(validation_path)
print(
    "SHA256:",
    file_sha256(validation_path)
)

print(closure_path)
print(
    "SHA256:",
    file_sha256(closure_path)
)


print("\nFINAL STATUS")
print("-" * 120)

print(
    f"Integrity checks passed : "
    f"{int(checks_df['passed'].sum())} / "
    f"{len(checks_df)}"
)

print("R2 status               : INTEGRITY VALIDATED")
print("Ready for packaging     : YES")

print("\n✓ R2-26 complete")
print("✓ Scientific results internally consistent")
print("✓ Publication tables validated")
print("✓ Frozen evidence unchanged")
print("✓ V1 retained for provenance")
print("✓ V2 authoritative")
print("✓ R2 ready for final packaging")

R2-26 — FINAL R2 INTEGRITY AND CONSISTENCY VALIDATION

REQUIRED ARTIFACT CHECK
------------------------------------------------------------------------------------------------------------------------
PASS     02_R2_AUDIT/R2_strict_feasibility_oracle_v2.py
PASS     02_R2_AUDIT/R2_legacy_kpi_coverage.csv
PASS     02_R2_AUDIT/R2_parser_truth_consistency.csv
PASS     02_R2_AUDIT/R2_parsed_slot_availability.csv
PASS     02_R2_AUDIT/R2_parsed_slot_reconstruction_provenance.json
PASS     02_R2_AUDIT/R2_E5_mapping_inspection.json
PASS     02_R2_AUDIT/R2_E5_exact_text_mapping_provenance.json
PASS     03_RESULTS_RAW/R2_reconstructed_parsed_slots.csv
PASS     03_RESULTS_RAW/R2_controlled_feasibility_cases_v2.csv
PASS     03_RESULTS_RAW/R2_per_intent_feasibility_audit_v2.csv
PASS     03_RESULTS_RAW/R2_strict_empty_cases_v2.csv
PASS     03_RESULTS_RAW/R2_mobility_only_disagreements_v2.csv
PASS     03_RESULTS_RAW/R2_E5_prediction_level_audit_v2.csv
PASS     03_RESULTS_RAW/R2_E5_false_feasible_decisi

In [102]:
# ============================================================
# R2-27 — FREEZE, MANIFEST, INVENTORY AND FINAL R2 PACKAGE
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
import shutil
import zipfile
from datetime import datetime

print("=" * 120)
print("R2-27 — FINAL R2 FREEZE AND EVIDENCE PACKAGE")
print("=" * 120)

ROOT = Path("/content/xagentic_r2_work")

AUDIT_DIR   = ROOT / "02_R2_AUDIT"
RAW_DIR     = ROOT / "03_RESULTS_RAW"
AGG_DIR     = ROOT / "04_RESULTS_AGGREGATE"
FINAL_DIR   = ROOT / "05_FINAL_EVIDENCE"
DOC_DIR     = ROOT / "06_DOCUMENTATION"
ARCHIVE_DIR = ROOT / "07_ARCHIVE"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# SHA helper
# ------------------------------------------------------------

def r2_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================
# 1. ARCHIVE SUPERSEDED V1 EVIDENCE
# ============================================================

print("\nARCHIVING SUPERSEDED V1 EVIDENCE")
print("-" * 120)

V1_ARCHIVE = (
    ARCHIVE_DIR /
    "R2_V1_LATENCY_ORACLE_SUPERSEDED"
)

V1_ARCHIVE.mkdir(
    parents=True,
    exist_ok=True
)

v1_files = [

    AUDIT_DIR /
    "R2_strict_feasibility_oracle.py",

    RAW_DIR /
    "R2_controlled_feasibility_cases.csv",

    RAW_DIR /
    "R2_per_intent_feasibility_audit.csv",

    AGG_DIR /
    "R2_feasibility_summary.csv",

    AGG_DIR /
    "R2_kpi_attribution.csv",

    RAW_DIR /
    "R2_strict_empty_cases.csv",

    RAW_DIR /
    "R2_mobility_only_disagreements.csv",
]

archived_v1 = []

for src in v1_files:

    if src.exists():

        dst = V1_ARCHIVE / src.name

        shutil.copy2(
            src,
            dst
        )

        archived_v1.append(dst)

        print(
            "ARCHIVED",
            src.name
        )

    else:

        print(
            "NOT FOUND",
            src.name
        )


# ------------------------------------------------------------
# V1 superseded README
# ------------------------------------------------------------

v1_readme = """R2 V1 STRICT ORACLE — SUPERSEDED EVIDENCE

STATUS
------

These files are retained only for methodological provenance.

They MUST NOT be used as the final R2 quantitative results.


REASON FOR SUPERSESSION
-----------------------

The V1 strict feasibility oracle evaluated latency using:

    requested_latency <= catalogue_max_latency

This does not correctly represent the directional semantics of a
maximum-latency SLA.

For a request of the form:

    latency <= X

the correct feasibility relationship is:

    catalogue_min_latency <= requested_max_latency


AUTHORITATIVE REPLACEMENT
-------------------------

R2 V2 corrects the latency direction while retaining strict evaluation of:

- latency
- throughput
- reliability
- density
- mobility

V2 also preserves an empty feasible set when no catalogue slice satisfies
the request.

The authoritative oracle is:

    R2_strict_feasibility_oracle_v2.py


PUBLICATION RULE
----------------

Do not report V1 R2-15 through R2-20 quantitative results.

Use only V2-controlled validation, the V2 1,200-intent audit, and the V2
100-decision E5 prediction-level audit for final R2 claims.

V1 is retained solely to document the complete experimental history.
"""

v1_readme_path = (
    V1_ARCHIVE /
    "SUPERSEDED_README.txt"
)

v1_readme_path.write_text(
    v1_readme,
    encoding="utf-8"
)


# ------------------------------------------------------------
# V1 archive manifest
# ------------------------------------------------------------

v1_manifest_rows = []

for p in sorted(
    V1_ARCHIVE.rglob("*")
):

    if p.is_file():

        v1_manifest_rows.append({
            "file":
                p.name,

            "size_bytes":
                p.stat().st_size,

            "sha256":
                r2_sha256(p),
        })

v1_manifest = pd.DataFrame(
    v1_manifest_rows
)

v1_manifest_path = (
    V1_ARCHIVE /
    "V1_ARCHIVE_SHA256_MANIFEST.csv"
)

v1_manifest.to_csv(
    v1_manifest_path,
    index=False
)

print(
    "\nV1 archive files:",
    len(v1_manifest)
)


# ============================================================
# 2. CREATE FINAL R2 README
# ============================================================

final_readme = """# XAgentic R2 — Strict Feasibility and Validity Audit

## Status

FINAL / FROZEN

R2 evaluates the original feasibility contract used by the frozen
XAgentic E1-E5 experiments.

The original experiment evidence was not modified.

No model was retrained.

No E5 prediction was regenerated.


## Authoritative oracle

R2_strict_feasibility_oracle_v2.py


## Core results

### Request-level audit

Frozen intents audited: 1,200

Legacy/strict candidate-set disagreements:
229 / 1,200 = 19.08%

Strict-empty requests:
14 / 1,200 = 1.17%

Legacy returned a non-empty candidate set for all 14 strict-empty requests.

Mobility contributed to over-acceptance:
61 / 1,200 = 5.08%

Mobility-only disagreements:
37 / 1,200 = 3.08%


### E5 prediction-level audit

Frozen E5 decisions audited: 100

Legacy-valid predictions:
100 / 100 = 100%

Strict-V2-valid predictions:
97 / 100 = 97%

False-feasible predictions:
3 / 100 = 3%

False-feasible + autonomous ACCEPT:
2 / 100 = 2%

False-feasible + ESCALATE:
1 / 100 = 1%


## Important interpretation

Two false-feasible predictions were classification-correct but violated
strict KPI feasibility.

Therefore classification correctness does not by itself guarantee safe
slice orchestration.

All three false-feasible E5 decisions corresponded to requests with an
empty strict feasible set.

Such requests require escalation, rejection, or explicit infeasibility
handling rather than selection of another catalogue slice.


## V1 provenance

The earlier V1 strict oracle is preserved under:

07_ARCHIVE/R2_V1_LATENCY_ORACLE_SUPERSEDED/

V1 used an incorrect latency-direction interpretation.

V1 results are retained only for provenance and MUST NOT be used for final
publication claims.


## Primary documentation

06_DOCUMENTATION/R2_FINAL_FINDINGS.md
06_DOCUMENTATION/R2_METHODOLOGY.md
06_DOCUMENTATION/R2_CLOSURE.json


## Evidence integrity

Frozen source, frozen corpus, and frozen E5 decision artifacts remained
unchanged throughout R2.

Final integrity validation passed 35/35 checks.
"""

final_readme_path = (
    ROOT /
    "R2_README_FINAL.md"
)

final_readme_path.write_text(
    final_readme,
    encoding="utf-8"
)


# ============================================================
# 3. CREATE COMPLETE ARTIFACT INVENTORY
# ============================================================

print("\nCREATING COMPLETE ARTIFACT INVENTORY")
print("-" * 120)

inventory_rows = []

for p in sorted(
    ROOT.rglob("*")
):

    if not p.is_file():
        continue

    rel = p.relative_to(ROOT)

    # Do not inventory an already-created final ZIP.
    if p.suffix.lower() == ".zip":
        continue

    rel_str = str(rel)

    if (
        "R2_V1_LATENCY_ORACLE_SUPERSEDED"
        in rel_str
    ):
        status = "SUPERSEDED_V1_PROVENANCE"

    elif (
        "_v2" in p.name.lower()
        or
        p.name in [
            "R2_FINAL_FINDINGS.md",
            "R2_METHODOLOGY.md",
            "R2_CLOSURE.json",
            "R2_FINAL_INTEGRITY_CHECKS.csv",
            "R2_publication_request_level_table.csv",
            "R2_publication_kpi_table.csv",
            "R2_publication_E5_decision_table.csv",
            "R2_publication_critical_cases.csv",
            "R2_README_FINAL.md",
        ]
    ):
        status = "AUTHORITATIVE_R2"

    else:
        status = "SUPPORTING_OR_FROZEN_EVIDENCE"

    inventory_rows.append({
        "relative_path":
            rel_str,

        "filename":
            p.name,

        "size_bytes":
            p.stat().st_size,

        "status":
            status,

        "sha256":
            r2_sha256(p),
    })


inventory = pd.DataFrame(
    inventory_rows
)

inventory_path = (
    FINAL_DIR /
    "R2_COMPLETE_ARTIFACT_INVENTORY.csv"
)

inventory.to_csv(
    inventory_path,
    index=False
)

print(
    "Files inventoried:",
    len(inventory)
)


# ============================================================
# 4. CREATE FINAL SHA256 MANIFEST
# ============================================================

print("\nCREATING SHA256 MANIFEST")
print("-" * 120)

manifest_rows = []

for p in sorted(
    ROOT.rglob("*")
):

    if not p.is_file():
        continue

    if p.suffix.lower() == ".zip":
        continue

    # Exclude manifest itself to avoid self-reference.
    if p.name == "R2_SHA256_MANIFEST.csv":
        continue

    manifest_rows.append({
        "relative_path":
            str(
                p.relative_to(ROOT)
            ),

        "size_bytes":
            p.stat().st_size,

        "sha256":
            r2_sha256(p),
    })


manifest = pd.DataFrame(
    manifest_rows
)

manifest_path = (
    FINAL_DIR /
    "R2_SHA256_MANIFEST.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

print(
    "Manifest entries:",
    len(manifest)
)


# ============================================================
# 5. FINAL FREEZE METADATA
# ============================================================

freeze = {

    "experiment":
        "R2 Strict Feasibility and Validity Audit",

    "status":
        "FINAL_FROZEN",

    "authoritative_oracle":
        "02_R2_AUDIT/R2_strict_feasibility_oracle_v2.py",

    "integrity_checks":
        "35/35 PASS",

    "request_level_results":
        {
            "n": 1200,
            "legacy_strict_disagreement": 229,
            "legacy_strict_disagreement_pct": 19.083,
            "strict_empty": 14,
            "strict_empty_pct": 1.167,
            "mobility_contributed": 61,
            "mobility_only": 37,
        },

    "e5_prediction_results":
        {
            "n": 100,
            "legacy_valid": 100,
            "strict_valid": 97,
            "false_feasible": 3,
            "false_feasible_accept": 2,
            "false_feasible_escalate": 1,
        },

    "v1":
        {
            "status":
                "SUPERSEDED_RETAINED_FOR_PROVENANCE",

            "archive":
                "07_ARCHIVE/R2_V1_LATENCY_ORACLE_SUPERSEDED",
        },

    "source_modified":
        False,

    "corpus_modified":
        False,

    "predictions_regenerated":
        False,

    "models_retrained":
        False,

    "ready_for_publication_use":
        True,
}

freeze_path = (
    FINAL_DIR /
    "R2_FINAL_FREEZE_METADATA.json"
)

with open(
    freeze_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        freeze,
        f,
        indent=2
    )


# ============================================================
# 6. CREATE FINAL ZIP
# ============================================================

ZIP_PATH = Path(
    "/content/"
    "XAgentic_R2_Strict_Feasibility_FINAL_2026-10-07.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("\nCREATING FINAL ZIP")
print("-" * 120)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as zf:

    for p in sorted(
        ROOT.rglob("*")
    ):

        if not p.is_file():
            continue

        # Do not recursively include ZIPs.
        if p.suffix.lower() == ".zip":
            continue

        arcname = (
            Path("XAgentic_R2_Strict_Feasibility_FINAL")
            /
            p.relative_to(ROOT)
        )

        zf.write(
            p,
            arcname=str(arcname)
        )


# ============================================================
# 7. VERIFY ZIP
# ============================================================

print("\nVERIFYING FINAL ZIP")
print("-" * 120)

assert ZIP_PATH.exists()
assert ZIP_PATH.stat().st_size > 0

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    bad = zf.testzip()

    assert bad is None, (
        f"STOP: corrupt ZIP member: {bad}"
    )

    names = zf.namelist()

print(
    "ZIP members:",
    len(names)
)

print(
    "ZIP size:",
    f"{ZIP_PATH.stat().st_size:,}",
    "bytes"
)

ZIP_SHA = r2_sha256(
    ZIP_PATH
)

print(
    "ZIP SHA256:",
    ZIP_SHA
)


# ------------------------------------------------------------
# Required ZIP-content checks
# ------------------------------------------------------------

required_zip_suffixes = [

    "R2_strict_feasibility_oracle_v2.py",
    "R2_per_intent_feasibility_audit_v2.csv",
    "R2_E5_prediction_level_audit_v2.csv",
    "R2_E5_false_feasible_decisions_v2.csv",
    "R2_publication_request_level_table.csv",
    "R2_publication_E5_decision_table.csv",
    "R2_FINAL_FINDINGS.md",
    "R2_METHODOLOGY.md",
    "R2_CLOSURE.json",
    "R2_FINAL_INTEGRITY_CHECKS.csv",
    "R2_COMPLETE_ARTIFACT_INVENTORY.csv",
    "R2_SHA256_MANIFEST.csv",
    "R2_FINAL_FREEZE_METADATA.json",
    "SUPERSEDED_README.txt",
    "R2_README_FINAL.md",
]

for suffix in required_zip_suffixes:

    matches = [
        n for n in names
        if n.endswith(suffix)
    ]

    assert matches, (
        f"STOP: required ZIP artifact missing: {suffix}"
    )

    print(
        "PASS",
        suffix
    )


# ============================================================
# 8. FINAL PACKAGE RECORD
# ============================================================

package_record = {
    "zip_filename":
        ZIP_PATH.name,

    "zip_size_bytes":
        ZIP_PATH.stat().st_size,

    "zip_sha256":
        ZIP_SHA,

    "zip_members":
        len(names),

    "zip_integrity":
        "PASS",

    "r2_status":
        "FINAL_FROZEN",

    "ready_for_download":
        True,
}

package_record_path = (
    FINAL_DIR /
    "R2_FINAL_PACKAGE_RECORD.json"
)

with open(
    package_record_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        package_record,
        f,
        indent=2
    )


print("\n" + "=" * 120)
print("R2 FINAL PACKAGE COMPLETE")
print("=" * 120)

print(
    "R2 status        : FINAL / FROZEN"
)

print(
    "Integrity        : 35 / 35 PASS"
)

print(
    "Final ZIP        :",
    ZIP_PATH
)

print(
    "ZIP SHA256       :",
    ZIP_SHA
)

print(
    "ZIP members      :",
    len(names)
)

print(
    "Ready to download: YES"
)

print("\n✓ R2 scientific work complete")
print("✓ V1 preserved and explicitly marked superseded")
print("✓ V2 authoritative evidence preserved")
print("✓ Complete inventory created")
print("✓ SHA256 manifest created")
print("✓ Final documentation preserved")
print("✓ Final ZIP integrity verified")

R2-27 — FINAL R2 FREEZE AND EVIDENCE PACKAGE

ARCHIVING SUPERSEDED V1 EVIDENCE
------------------------------------------------------------------------------------------------------------------------
ARCHIVED R2_strict_feasibility_oracle.py
ARCHIVED R2_controlled_feasibility_cases.csv
ARCHIVED R2_per_intent_feasibility_audit.csv
ARCHIVED R2_feasibility_summary.csv
ARCHIVED R2_kpi_attribution.csv
ARCHIVED R2_strict_empty_cases.csv
ARCHIVED R2_mobility_only_disagreements.csv

V1 archive files: 8

CREATING COMPLETE ARTIFACT INVENTORY
------------------------------------------------------------------------------------------------------------------------
Files inventoried: 115

CREATING SHA256 MANIFEST
------------------------------------------------------------------------------------------------------------------------
Manifest entries: 116

CREATING FINAL ZIP
------------------------------------------------------------------------------------------------------------------------

VERIFYIN

In [103]:
from google.colab import files

files.download(
    "/content/XAgentic_R2_Strict_Feasibility_FINAL_2026-10-07.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>